# Oddball information across areas — LFP spectral MI

**OpenScope Community Predictive Processing** · DANDI `001637`

> **Before you run it.** Replicating the analysis from scratch streams roughly 60 sessions
> from DANDI and takes **several hours**; stages 1–2 dominate. It writes **about 400 MB** of
> intermediate data (per-animal trial PSDs) plus the figure panels. Everything is resumable:
> re-running skips whatever is already saved. If you only want the figures, the panels and
> the interactive HTML in `publication/` are enough; no recomputation is needed to browse them.

How much information does the LFP power spectrum carry about each prediction error, and
in which areas and frequencies? The notebook runs the same analysis in every session
context listed in `RUN_CONTEXTS` (all four by default), in one pass:

| Context | Block | Oddballs |
|---|---|---|
| `standard` | Standard mismatch block (passive) | `orientation_45`, `orientation_90`, `halt`, `omission` |
| `sensorimotor` | Sensory-motor mismatch block (closed loop: running drives the grating) | `motor_orientation_45`, `motor_orientation_90`, `motor_halt`, `motor_omission` |
| `sequence` | Sequence mismatch block (90°, 45°, 0°, 45°, blank, repeating) | `orientation_45`, `orientation_90`, `halt`, `omission` in the 3rd slot |
| `duration` | Duration mismatch block (timing deviants) | early/late gratings, classified from the trial timing (`isi_150ms`, `isi_500ms`, `isi_1000ms`), plus `omission` |

**Run all cells once, top to bottom.** Every stage loops over the contexts, and each
context writes to its own results folder. Anything already saved is skipped, so a
re-run only computes what is missing. Section 6 renders the publication figure, one per
paradigm (about 20 panel PNGs plus the composites), and section 7 writes the interactive
version, the same figure with a paradigm switch, as one self-contained HTML file.

For every animal the notebook:

1. streams the session and reads the trial table of the context's block;
2. randomly samples up to **10 LFP channels per main area**;
3. cuts a **1 s chunk centred on each trial** (35 trials per oddball type, up to 400
   standards; for timing deviants, centred on the moment the violation happens, see
   below) and computes its PSD with a **Gaussian taper whose FWHM equals the stimulus
   duration**, averaged into 50 log-spaced bins from 4 to 100 Hz;
4. computes the **mutual information** between log-power and trial identity (standard vs.
   each oddball), per area and per frequency, together with the signed **Δ power** (dB);
5. (stage 1b) reads the **matched control** trials, the same stimuli in a control block
   with no regularity to violate (randomly ordered blocks, standard and sequence paradigms),
   and tops up **V1 with channels from every cortical
   layer**, so the same MI and Δ power can be computed for *oddball vs control*,
   *standard vs control*, and per V1 layer.

The publication figure shows, for each paradigm: **A** an example V1 trial with its
trial-averaged spectrogram, **B** the V1 power spectrum per condition, **C** V1 spectra next
to the same stimuli in the control block, and **D–E** heatmaps over all areas and
frequencies (one column per oddball type) of the power difference vs the standard and vs
the matched control, with dots where the MI is significant across animals.

**Changes after the forum feedback** (Sarruedi, Maier): a plain-language explanation of
the MI (next cell) and a single-trial panel that shows where it comes from; Δ power next to
every MI heatmap; the matched-control contrast, which keeps the stimulus and its
orientation tuning fixed and removes adaptation to the standard, plus standard vs control
to show that adaptation; V1 by cortical layer; and examples chosen by a fixed rule so the
modulation is clear but typical (section 6).

**Main areas** follow Table 1 of the project manuscript (Neuropixels cohorts: V1, LM, M1,
M2, PL/IL/ACA, RL, LGN) plus the striatum (CP) and the hippocampal and thalamic structures
the probes pass through (CA1, CA2, CA3, DG, SUB, LP). Edit `MAIN_AREAS` to change this.

**Persistence.** Results are written as they are produced: to Google Drive when running on
Colab (optional, on by default), otherwise to a local folder. One file of trial PSDs per
animal, then one file of MI per animal. Both stages skip animals already saved, so the
notebook can be interrupted and resumed, and the MI can be recomputed with new parameters
without streaming the data again. The last cell writes `viewer.html`, a standalone web page
version of the interactive figure.

**Design choices worth knowing about**
- *MI estimator*: Gaussian-copula MI (Ince et al., 2017, *Hum Brain Mapp*). Rank-based, so
  independent of LFP units or scaling, and closed-form, so hundreds of shuffles are cheap.
- *Bias*: each value is corrected by subtracting the mean MI of label-shuffled data; the
  shuffles also give a per-animal p-value.
- *Balance*: every estimate uses equal numbers of standards and oddballs (standards redrawn
  `N_DRAWS` times and averaged). With balanced classes the ceiling is 1 bit.
- *Standards, standard context*: the first `SKIP_AFTER_DEVIANT` standards after any oddball
  are excluded, so the standard class is not contaminated by post-oddball recovery.
- *Standards, sensorimotor context*: the closed-loop "standard" rows are 33 ms display
  frames of continuous, wheel-coupled flow, not discrete presentations. Standards are
  therefore **pseudo-trials** at random times in the block, well away from any oddball.
  Running changes the LFP on its own (hippocampal theta above all), so the pseudo-trials are
  **matched to the oddballs on running speed**, and in every MI draw each oddball type gets
  its own speed-matched set of standards.
- *Standards, sequence context*: only standards in the **3rd slot** of the sequence (the
  slot where oddballs occur, normally 0°) are used, and the whole sequence after an oddball
  is skipped. Note that a 45° or 90° grating in that slot is not a new stimulus — both
  orientations appear elsewhere in every sequence — so here orientation oddballs are
  violations of *order*, not of feature novelty.
- *What the duration block is*: every grating is the same (0°, 343 ms). What varies is
  the gray gap before a grating: 343 ms for standards, and 150 ms (the grating comes
  early), 500 ms or 1000 ms (it comes late) for the deviants, which the table labels only
  as `jitter`; the notebook names them from their measured gap. Omissions are the fourth
  oddball. Halt only occurs in the control blocks.
- *Alignment, duration context*: each chunk is centred on the moment the violation becomes
  detectable, i.e. whichever comes first of the actual event and the expected one. A grating
  that is **too short** is centred on its (early) offset; one that is **too long** on the
  expected offset, when it fails to turn off. A gray interval that is **too short** is
  centred on the early onset of the grating; one that is **too long** on the expected
  onset, when nothing appears yet (with a 500 ms gap the late grating still falls inside
  the taper; with 1000 ms it does not). Standards are centred on the same event (their offset or the
  next onset), so each contrast compares like with like. Omissions stay stimulus-centred. Without this, any difference in running
  between the two classes would show up as MI that has nothing to do with the mismatch.

### What the mutual information means here

Take one area and one frequency band, and look at the LFP power on every single trial. If
oddball trials tend to have more (or less) power than standard trials, then the power of a
single trial tells you something about which kind of trial it was. The **mutual
information (MI)** measures how much, in bits:

- **0 bits**: the power on oddball and standard trials has the same distribution; one
  trial's power says nothing about whether an oddball happened.
- **1 bit**: the distributions do not overlap; one trial is enough to tell them apart
  perfectly. With equal numbers of each kind of trial (as here), 1 bit is the maximum.
- In between, for roughly Gaussian distributions: **0.04 bits ≈ d′ 0.5** (a single trial
  classified ≈60 % correctly), **0.16 bits ≈ d′ 1** (≈69 %), **0.5 bits ≈ d′ 2** (≈84 %).

Biologically, the LFP in a band mostly reflects synchronized synaptic currents of many
nearby neurons, so the MI measures how *reliably*, trial after trial, that population
activity changes when the expectation is violated, not only whether its average changes. A
large average change that varies a lot from trial to trial carries little information; a
modest but consistent change carries more. Unlike an average, MI also captures changes in
the spread of the distribution and is unaffected by the units of the LFP.

**MI has no sign**: an increase and a decrease of power both give positive MI. That is why
every MI heatmap has a **Δ power** row underneath (oddball minus reference, in dB), and why
the *single-trial* panel of the figure shows the two distributions whose separation the MI
quantifies.

**What the contrast isolates.** Oddball vs standard mixes several things that need no
prediction: the oddball is a different stimulus, so it drives a different set of
orientation-tuned neurons, and the standard has been repeated many times, so the neurons
that respond to it are adapted while the oddball's are not (*release from adaptation*). The
**control block** shows the same stimuli in random order with the same timing.
*Oddball vs control* keeps the stimulus, and therefore its orientation tuning, identical
(the control trials have the same orientation and drift direction, read from the trial
table) and compares it with a context with no regularity to violate: what is left is the
effect of the violated expectation. *Standard vs control* shows the adaptation itself.
This is the decomposition used in the stimulus-specific adaptation literature into
prediction error (oddball − control) and repetition suppression (control − standard), e.g.
Parras et al., 2017, *Nat Commun*. Two caveats: in Control block 1 each orientation appears on
~6 % of trials, more often than an oddball does in its block, and the control block
is recorded at a different time in the session, so slow changes of state can contribute.

## 0 · Setup

In [ ]:
#@title Install dependencies
!pip install -q dandi pynwb remfile

In [ ]:
#@title Storage — Google Drive if running on Colab (optional), otherwise a local folder
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE    = True                           #@param {type:"boolean"}
DRIVE_SUBDIR = "OpenScope_PP/oddball_MI"      #@param {type:"string"}
LOCAL_DIR    = "./oddball_MI"                 #@param {type:"string"}

# On Colab with USE_DRIVE, results persist on Google Drive (a Colab runtime's own disk is
# wiped when it disconnects). Anywhere else -- or with USE_DRIVE off -- they go to LOCAL_DIR.
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = os.path.join("/content/drive/MyDrive", DRIVE_SUBDIR)
else:
    RESULTS_DIR = os.path.abspath(LOCAL_DIR)
    if IN_COLAB:
        print("Drive is off: results live on this Colab runtime and are lost when it disconnects.")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results: {RESULTS_DIR}")

In [ ]:
#@title Imports and configuration
import glob
import json
import logging
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import periodogram
from scipy.special import ndtri
from scipy.stats import rankdata, wilcoxon

warnings.filterwarnings("ignore")
logging.getLogger("dandi").setLevel(logging.CRITICAL)   # silences harmless 403 HEAD noise

# ── Context ------------------------------------------------------------------
# Every stage below runs over all of these; remove any you do not want.
RUN_CONTEXTS = ["standard", "sensorimotor", "sequence", "duration"]

CONTEXTS = {
    "standard": {
        "label": "Standard mismatch",
        "table_regex": r"^Standard mismatch block_presentations$",
        "standard": "standard",
        "oddballs": ["orientation_45", "orientation_90", "halt", "omission"],
        "standards_from": "presentations",     # discrete standard presentations
        "match_running": False,
    },
    "sensorimotor": {
        "label": "Sensorimotor mismatch (closed loop)",
        "table_regex": r"(?i)^(sensory.?motor|motor).*(mismatch|oddball).*_presentations$",
        "standard": "standard",
        "oddballs": ["motor_orientation_45", "motor_orientation_90", "motor_halt",
                     "motor_omission"],
        "standards_from": "pseudo",            # standard rows are 33 ms closed-loop frames
        "match_running": True,
    },
    "sequence": {
        "label": "Sequence mismatch",
        "table_regex": r"(?i)^sequen\w*.*(mismatch|oddball).*_presentations$",
        "standard": "standard",
        "oddballs": ["orientation_45", "orientation_90", "halt", "omission"],
        "standards_from": "presentations",
        "standard_position": 3,                # only standards in the oddball slot (normally 0°)
        "skip_after": 5,                       # one whole sequence after any oddball
        "match_running": False,
    },
    "duration": {
        "label": "Duration mismatch",
        "table_regex": r"(?i)^(duration|jitter|temporal|timing)\w*.*(mismatch|oddball).*_presentations$",
        "standard": "standard",
        "oddballs": None,                      # classified from trial timing (TIMING_TOL)
        "psd_version": 2,                      # v2: gray interval before the grating is checked
        "standards_from": "presentations",
        "match_running": False,
    },
}
TRIALTYPE_COLS = ("TrialType", "Trial_Type", "trial_type")

BASE_LABELS = {"orientation_45": "Orientation 45°", "orientation_90": "Orientation 90°",
               "halt": "Halt", "omission": "Omission"}


def base_of(oddball):
    return re.sub(r"^motor_", "", oddball)


def label_of(oddball):
    b = base_of(oddball)
    if b == "standard":
        return "Standard (adaptation)"
    if b in BASE_LABELS:
        return BASE_LABELS[b]
    m = re.match(r"(duration|isi)_(\d+)ms$", b)
    if m:
        ms = int(m.group(2))
        if m.group(1) == "duration":
            return f"Grating {ms} ms ({'short' if ms < 343 else 'long'})"
        return f"Gap {ms} ms ({'early' if ms < 343 else 'late'})"
    return b


def order_oddballs(names):
    # Timing deviants by kind and length, then the feature oddballs in their usual order.
    def key(n):
        b = base_of(n)
        m = re.match(r"(duration|isi)_(\d+)ms$", b)
        if m:
            return (0 if m.group(1) == "duration" else 1, int(m.group(2)))
        if b in BASE_LABELS:
            return (2, list(BASE_LABELS).index(b))
        return (3, 0)
    return sorted({str(n) for n in names}, key=key)


# ── Dataset ------------------------------------------------------------------
DANDISET_ID      = "001637"
DANDISET_VERSION = "draft"      # switch to a published version once one exists

# ── Main areas: display name -> regex on the electrode `location` (Allen CCF acronym)
# Anchored patterns keep sibling areas apart: VISp does not swallow VISpm/VISpl/VISpor,
# VISl does not swallow VISli. Order here is the order on the figure's x axis.
MAIN_AREAS = {
    "LGN":  r"^LGd",
    "V1":   r"^VISp(\d|$)",
    "LM":   r"^VISl(\d|$)",
    "RL":   r"^VISrl",
    "AL":   r"^VISal",
    "PM":   r"^VISpm",
    "AM":   r"^VISam",
    "M1":   r"^MOp(\d|$)",
    "M2":   r"^MOs(\d|$)",
    "ACA":  r"^ACA",
    "PL":   r"^PL(\d|$)",
    "ILA":  r"^ILA",
    "CP":   r"^(CP|STR)$",   # dorsal striatum (caudoputamen); bare STR if unassigned
    "LP":   r"^LP$",
    "CA1":  r"^CA1$",
    "CA2":  r"^CA2$",
    "CA3":  r"^CA3$",
    "DG":   r"^DG",
    "SUB":  r"^(SUB|ProS)",
}
AREA_ORDER = list(MAIN_AREAS)

# ── Channel sampling ---------------------------------------------------------
MAX_CH_PER_AREA = 10       # random channels per main area (all of them if fewer)
RNG_SEED        = 42       # combined with the subject id, so each animal is reproducible

# ── Trials -------------------------------------------------------------------
MAX_STANDARDS      = 400   # standards (or pseudo-standards) kept per animal
N_PER_ODDBALL      = 35    # random trials kept per oddball type
SKIP_AFTER_DEVIANT = 2     # standards right after an oddball are excluded (sequence: 5)

# Duration context: how a deviant's timing is classified
TIMING_TOL  = 0.025        # s; a duration or gray interval this far from the standard's is a deviant
TIMING_STEP = 0.05         # s; deviant lengths are rounded to this for naming (absorbs frame jitter)

# Sensorimotor context: pseudo-standards and running speed
PSEUDO_CANDIDATES = 4000   # random candidate times drawn before exclusion and matching
EXCLUDE_BEFORE    = 0.5    # s; no pseudo-standard chunk may come this close before an oddball
EXCLUDE_AFTER     = 2.0    # s; ...or this close after one ends
RUNNING_SERIES    = None   # e.g. "running/running_speed"; None -> auto-detect (see section 2)
MIN_SPEED         = None   # cm/s; e.g. 2.0 keeps only running trials, in both classes

# ── Chunks and PSD -----------------------------------------------------------
CHUNK_DUR   = 1.0          # s, centred on the middle of the stimulus
TAPER_FWHM  = None         # s; None -> median standard duration of each session (~0.35 s)
# The Gaussian taper is what sets the effective window: at the edges of the stimulus it
# weighs 0.5, and at the neighbouring presentations (~0.7 s away) it is below 0.5 %, so
# they do not leak in even though the 1 s chunk reaches them.
FMIN, FMAX   = 4.0, 100.0
N_FREQ_BINS  = 50          # log-spaced bins; power is averaged within each bin
NFFT         = 16384       # zero-padded FFT so even the narrowest low bin holds several points
FREQ_EDGES   = np.geomspace(FMIN, FMAX, N_FREQ_BINS + 1)
FREQ_CENTERS = np.sqrt(FREQ_EDGES[:-1] * FREQ_EDGES[1:])   # geometric centres
# Note: a Gaussian taper with FWHM ~0.35 s resolves ~1.8 Hz (FWHM of its power kernel).
# Log bins are narrower than that below ~25 Hz, so there neighbouring bins oversample
# the same spectral information rather than being independent estimates.

# ── Reading ------------------------------------------------------------------
MAX_READ_SPAN_S = 60.0     # trials are read in batches no longer than this
MAX_READ_GAP_S  = 3.0      # ...and a gap longer than this starts a new batch

# ── Mutual information -------------------------------------------------------
N_DRAWS    = 20            # balanced redraws of the larger class
N_SHUFFLE  = 200           # label shuffles per draw (bias correction + p-values)
MIN_TRIALS = 20            # per class; fewer and the area/oddball is skipped

# ── Group statistics ---------------------------------------------------------
MIN_ANIMALS_SHOW  = 3      # areas recorded in fewer animals are not plotted
MIN_ANIMALS_STATS = 6      # one-sided Wilcoxon needs >= 6 animals to reach p < 0.05
ALPHA             = 0.05   # FDR (Benjamini-Hochberg) across area x frequency, per oddball

MI_TAG = f"gcmi_d{N_DRAWS}_s{N_SHUFFLE}_c{int(CHUNK_DUR*1000)}ms_gauss_b{N_FREQ_BINS}log"
MI_VERSION = 2             # v2 adds Δ power, the matched-control contrast and V1 layers

# ── Matched controls --------------------------------------------------------------
# The same stimulus shown inside a block with no regularity to violate. The standard and
# sequence blocks each have one (Control block 1.x: gratings at many orientations, halts and
# omissions at random, same timing as the standard block; Control block 2: the sequence gratings
# shuffled). Oddball vs control keeps the stimulus, and therefore orientation tuning, fixed,
# and leaves out adaptation to the standard. Standard vs control measures that adaptation.
# Control blocks are found by their BlockType column (the table name is the fallback).
# Sensorimotor and duration only get the oddball-vs-standard contrast: the open-loop replay
# block presents the sensorimotor events only ~8 times per type, too few for a control, and
# no control block has the duration paradigm's gap deviants.
# Optional keys: same_type (control rows carry the event's own TrialType; else gratings are
# "single"), include_standard (also compare the standard with its control; default True),
# min_trials (minimum per class for this contrast; default MIN_TRIALS).
CONTROLS = {
    "standard": {"block_type": r"(?i)^standard_control$",
                 "table_regex": r"(?i)^control block 1(\.\d+)?_presentations$"},
    "sequence": {"block_type": r"(?i)^sequen\w*_control",
                 "table_regex": r"(?i)^control block 2_presentations$"},
}
MAX_CONTROL_PER_TYPE = 70  # control trials kept per stimulus (each MI draw balances the classes)


def control_type(cond, same_type=False):
    # TrialType of the same stimulus in a control block: gratings are "single" there, unless
    # the control block keeps the event's own label (the sensorimotor open-loop replay).
    if same_type:
        return cond
    b = base_of(cond)
    return "single" if (b == "standard" or b.startswith("orientation_")) else b


# ── V1 by cortical layer -------------------------------------------------------------
# Stage 1b tops up V1 with channels from every layer (the main sample is 10 random V1
# channels, so a layer can be missing), then the MI is also computed per layer.
LAYERS = {"L1": r"^VISp1$", "L2/3": r"^VISp2/3$", "L4": r"^VISp4$",
          "L5": r"^VISp5$", "L6": r"^VISp6[ab]?$"}
LAYER_ORDER = list(LAYERS)
MAX_CH_PER_LAYER = 5       # V1 channels per layer, counting the ones already sampled
LAYER_CONTEXTS = list(RUN_CONTEXTS)   # topping up re-reads each session once; [] skips it


def layer_of(location):
    for name, pattern in LAYERS.items():
        if re.match(pattern, str(location)):
            return name
    return None


# ── Folders: the standard context keeps the original layout, so earlier results load as-is
def context_dirs(ctx):
    base = RESULTS_DIR if ctx == "standard" else os.path.join(RESULTS_DIR, ctx)
    return {k: os.path.join(base, k) for k in ("psd", "mi", "figures")}


for _c in RUN_CONTEXTS:
    for d in context_dirs(_c).values():
        os.makedirs(d, exist_ok=True)
print("Contexts to run:", RUN_CONTEXTS)

In [ ]:
#@title Sessions
# Standard context: the same 15 sessions as the VISp2/3 showcase notebook, hard-coded so
# the figure does not change as DANDI 001637 grows.
STANDARD_SESSION_PATHS = [
    "sub-834687/sub-834687_ses-ecephys-834687-2026-03-18-15-50-10_ecephys.nwb",
    "sub-830851/sub-830851_ses-ecephys-830851-2026-03-17-10-46-04_ecephys.nwb",
    "sub-830848/sub-830848_ses-ecephys-830848-2026-03-04-15-34-54_ecephys.nwb",
    "sub-834691/sub-834691_ses-ecephys-834691-2026-02-17-10-49-35_ecephys.nwb",
    "sub-830795/sub-830795_ses-ecephys-830795-2026-02-24-15-08-59_ecephys.nwb",
    "sub-830794/sub-830794_ses-ecephys-830794-2026-01-27-11-25-31_ecephys.nwb",
    "sub-830846/sub-830846_ses-ecephys-830846-2026-03-11-10-19-32_ecephys.nwb",
    "sub-830847/sub-830847_ses-ecephys-830847-2026-03-11-15-46-40_ecephys.nwb",
    "sub-830849/sub-830849_ses-ecephys-830849-2026-03-06-10-51-58_ecephys.nwb",
    "sub-830852/sub-830852_ses-ecephys-830852-2026-02-24-10-20-58_ecephys.nwb",
    "sub-832691/sub-832691_ses-ecephys-832691-2026-03-25-10-22-33_ecephys.nwb",
    "sub-834686/sub-834686_ses-ecephys-834686-2026-03-25-15-47-48_ecephys.nwb",
    "sub-848387/sub-848387_ses-ecephys-848387-2026-05-05-15-55-50_ecephys.nwb",
    "sub-848390/sub-848390_ses-ecephys-848390-2026-05-05-10-08-25_ecephys.nwb",
    "sub-820454/sub-820454_ses-ecephys-820454-2025-11-05-15-21-15_ecephys.nwb",
]

# Other contexts: section 2 scans the animals' other NWB files once and stores the list of
# tables in each one (an inventory in the results folder). Each context's sessions are then derived from
# that inventory with the context's block-name pattern, so a pattern can be fixed without
# re-scanning, and a context that matched nothing is reported instead of silently skipped.
INVENTORY_FILE = os.path.join(RESULTS_DIR, "nwb_tables_inventory.json")


def subject_of(path):
    return path.split("/")[0].replace("sub-", "")


def load_inventory():
    if not os.path.exists(INVENTORY_FILE):
        return None
    with open(INVENTORY_FILE) as f:
        return json.load(f)


def load_sessions(ctx):
    # Session paths for a context (one per animal that has the block); None before the scan.
    if ctx == "standard":
        return list(STANDARD_SESSION_PATHS)
    inv = load_inventory()
    if inv is None:
        return None
    regex = CONTEXTS[ctx]["table_regex"]
    out = []
    for p in STANDARD_SESSION_PATHS:
        subj = subject_of(p)
        hits = [f for f in sorted(inv) if f.startswith(f"sub-{subj}/")
                and any(re.search(regex, t) for t in inv[f])]
        if hits:
            out.append(hits[0])
    return out


def set_context(ctx):
    # Point every stage at one context: its settings, its folders and its sessions.
    global CONTEXT, CTX, STANDARD, ODDBALLS, PSD_DIR, MI_DIR, FIG_DIR, SESSION_PATHS
    CONTEXT, CTX = ctx, CONTEXTS[ctx]
    STANDARD, ODDBALLS = CTX["standard"], CTX["oddballs"]
    d = context_dirs(ctx)
    PSD_DIR, MI_DIR, FIG_DIR = d["psd"], d["mi"], d["figures"]
    SESSION_PATHS = load_sessions(ctx)
    return SESSION_PATHS


def banner(ctx):
    print(f"\n{'=' * 78}\n{CONTEXTS[ctx]['label']}\n{'=' * 78}")


for _c in RUN_CONTEXTS:
    _s = load_sessions(_c)
    print(f"{_c:>13}: " + (f"{len(_s)} sessions" if _s is not None
                           else "not scanned yet (section 2 finds them)"))

## 1 · Helpers

In [ ]:
#@title Streaming, electrodes and areas
def stream_nwb(filepath):
    # Open an NWB file straight from the DANDI S3 bucket; nothing is downloaded to disk.
    import h5py
    import remfile
    from dandi.dandiapi import DandiAPIClient
    from pynwb import NWBHDF5IO
    with DandiAPIClient() as client:
        asset = client.get_dandiset(DANDISET_ID, DANDISET_VERSION).get_asset_by_path(filepath)
        url = asset.get_content_url(follow_redirects=True, strip_query=True)
    io = NWBHDF5IO(file=h5py.File(remfile.File(url), "r"), mode="r")
    return io, io.read()


def list_nwb_assets():
    # Every NWB path in the dandiset.
    from dandi.dandiapi import DandiAPIClient
    with DandiAPIClient() as client:
        ds = client.get_dandiset(DANDISET_ID, DANDISET_VERSION)
        return sorted(a.path for a in ds.get_assets() if a.path.endswith(".nwb"))


def find_table(nwb, regex=None):
    # Name of the intervals table for this context (None if the session does not have it).
    regex = regex or CTX["table_regex"]
    for name in nwb.intervals.keys():
        if re.search(regex, name):
            return name
    return None


def lfp_series(nwb):
    return nwb.processing["ecephys"]["LFP"].electrical_series


def area_of(location):
    # Main area for a CCF acronym, or None if it is not one of MAIN_AREAS.
    for area, pattern in MAIN_AREAS.items():
        if re.match(pattern, location):
            return area
    return None


def lfp_channels(nwb):
    # One row per electrode that carries LFP: id, location, main area, series and column.
    # LFP is spatially subsampled, so the electrodes table alone is not a channel list.
    edf = nwb.electrodes.to_dataframe().reset_index()
    edf = edf.rename(columns={edf.columns[0]: "electrode_id"})
    loc = dict(zip(edf["electrode_id"].astype(int), edf["location"].astype(str), strict=True))

    rows = []
    for name, es in lfp_series(nwb).items():
        for col, eid in enumerate(es.electrodes.data[:]):
            eid = int(eid)
            rows.append({"electrode_id": eid, "location": loc.get(eid, ""),
                         "series": name, "col": col})
    df = pd.DataFrame(rows)
    df["area"] = df["location"].map(area_of)
    return df


def sample_channels(chans, max_per_area, rng):
    # Up to max_per_area random LFP channels from each main area present.
    pool = chans.dropna(subset=["area"])
    picked = []
    for a in AREA_ORDER:
        sub = pool[pool["area"] == a]
        if len(sub):
            picked.append(sub.sample(n=min(max_per_area, len(sub)),
                                     random_state=int(rng.integers(2**31))))
    return pd.concat(picked).reset_index(drop=True) if picked else pool.iloc[:0]

In [ ]:
#@title Running speed
def timeseries_in(nwb):
    # Every object with data in processing modules and acquisition, keyed "module/name".
    out = {}
    for mod_name, mod in nwb.processing.items():
        for name, obj in getattr(mod, "data_interfaces", mod).items():
            if hasattr(obj, "data"):
                out[f"{mod_name}/{name}"] = obj
    for name, obj in getattr(nwb, "acquisition", {}).items():
        if hasattr(obj, "data"):
            out[f"acquisition/{name}"] = obj
    return out


def running_speed(nwb):
    # (timestamps, speed) of the running-speed series, or None if there is none.
    series = timeseries_in(nwb)
    if RUNNING_SERIES:
        name = RUNNING_SERIES
    else:
        cands = [k for k in series if re.search(r"(?i)speed", k.split("/")[-1])]
        cands.sort(key=lambda k: (not re.search(r"(?i)running", k), len(k)))
        if not cands:
            return None
        name = cands[0]
    ts_obj = series[name]
    v = np.asarray(ts_obj.data[:], dtype=float).ravel()
    if getattr(ts_obj, "timestamps", None) is not None:
        t = np.asarray(ts_obj.timestamps[:], dtype=float)
    else:
        t = ts_obj.starting_time + np.arange(len(v)) / float(ts_obj.rate)
    return t, v


def window_speed(speed, t0, t1):
    # Mean speed in each window [t0, t1); NaN if no speed data (or no samples) there.
    t0, t1 = np.atleast_1d(t0), np.atleast_1d(t1)
    if speed is None:
        return np.full(len(t0), np.nan)
    t, v = speed
    cs = np.r_[0.0, np.cumsum(np.nan_to_num(v))]
    i0, i1 = np.searchsorted(t, t0), np.searchsorted(t, t1)
    n = i1 - i0
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(n > 0, (cs[i1] - cs[i0]) / np.maximum(n, 1), np.nan)


def match_by_speed(cand_speed, odd_speed, n_total, rng):
    # Indices of candidates matched to the oddballs on speed (nearest neighbours, no reuse).
    # Every oddball claims the same number of candidates, so the matched standards inherit
    # the oddballs' speed distribution.
    k = int(np.ceil(n_total / max(1, np.isfinite(odd_speed).sum())))
    free = np.isfinite(cand_speed)
    chosen = []
    for s in rng.permutation(odd_speed[np.isfinite(odd_speed)]):
        d = np.where(free, np.abs(cand_speed - s), np.inf)
        pick = np.argsort(d)[:k]
        pick = pick[np.isfinite(d[pick])]
        free[pick] = False
        chosen.extend(pick.tolist())
    return np.array(chosen[:n_total], dtype=int)

In [ ]:
#@title Trial table
def trial_type_col(df):
    for c in TRIALTYPE_COLS:
        if c in df.columns:
            return c
    raise KeyError(f"no trial-type column among {TRIALTYPE_COLS}: {list(df.columns)}")


def standard_mask(df, types):
    # Standard presentations eligible as standards (sequence context: 3rd slot only).
    is_std = types == STANDARD
    pos = CTX.get("standard_position")
    if pos is None:
        return is_std
    pos_col = next((c for c in df.columns
                    if re.search(r"(?i)trial.?in.?sequence|sequence.?position", c)), None)
    if pos_col is not None:
        return is_std & (pd.to_numeric(df[pos_col], errors="coerce").to_numpy() == pos)
    ori_col = next((c for c in df.columns if c.lower() == "orientation"), None)
    if ori_col is None:
        raise KeyError(f"no sequence-position or orientation column: {list(df.columns)}")
    # In the 90-45-0-45 sequence, 0° only occurs in the 3rd slot (0 rad == 0°).
    print("  ! no sequence-position column — using the 0° standards, which only occur in slot 3")
    ori = pd.to_numeric(df[ori_col], errors="coerce").to_numpy()
    return is_std & (np.abs(ori) < 1e-3)


def oddball_classes(df, types):
    # Oddball class (None for non-oddballs) and anchor for every row, plus the standard's
    # duration D0 and gray interval G0. Anchors: "centre" (stimulus middle), "offset",
    # "onset" (this grating's onset), "next_onset" (onset of the following grating).
    t0, t1 = df["start_time"].to_numpy(), df["stop_time"].to_numpy()
    nxt = np.r_[t0[1:], np.nan]
    dur, gap = t1 - t0, nxt - t1
    gap_before = t0 - np.r_[np.nan, t1[:-1]]
    is_std = types == STANDARD
    D0 = float(np.nanmedian(dur[is_std]))
    G0 = float(np.nanmedian(gap[is_std & np.r_[is_std[1:], False]]))
    cls = np.full(len(df), None, dtype=object)
    anchor = np.full(len(df), "centre", dtype=object)

    if CTX["oddballs"] is not None:
        hit = np.isin(types, CTX["oddballs"])
        cls[hit] = types[hit]
        return cls, anchor, D0, G0

    def name(x):
        return int(round(x / TIMING_STEP) * TIMING_STEP * 1000)

    for i in np.flatnonzero(~is_std):
        raw = types[i]
        if base_of(raw) in BASE_LABELS:                 # e.g. omission: keep, stimulus-centred
            cls[i] = raw
        elif abs(dur[i] - D0) > TIMING_TOL:
            cls[i], anchor[i] = f"duration_{name(dur[i])}ms", "offset"
        elif np.isfinite(gap_before[i]) and abs(gap_before[i] - G0) > TIMING_TOL:
            # The gray interval before this grating was off: it came early or late.
            cls[i], anchor[i] = f"isi_{name(gap_before[i])}ms", "onset"
        elif np.isfinite(gap[i]) and abs(gap[i] - G0) > TIMING_TOL:
            cls[i], anchor[i] = f"isi_{name(gap[i])}ms", "next_onset"
        elif not raw.startswith("sequence_"):
            cls[i] = f"{raw}_unclassified"
    return cls, anchor, D0, G0


def anchor_times(df, idx, kind, D0, G0, is_oddball):
    # Time each chunk is centred on. Oddballs: the earlier of the actual and the expected
    # event (when the violation becomes detectable). Standards: the actual event.
    t0, t1 = df["start_time"].to_numpy()[idx], df["stop_time"].to_numpy()[idx]
    nxt = np.r_[df["start_time"].to_numpy()[1:], np.nan][idx]
    prev_stop = np.r_[np.nan, df["stop_time"].to_numpy()[:-1]][idx]
    if kind == "centre":
        return 0.5 * (t0 + t1)
    if kind == "offset":
        return t0 + np.minimum(t1 - t0, D0) if is_oddball else t1
    if kind == "onset":
        return prev_stop + np.minimum(t0 - prev_stop, G0) if is_oddball else t0
    if kind == "next_onset":
        return t1 + np.minimum(nxt - t1, G0) if is_oddball else nxt
    raise ValueError(kind)


def build_trials(nwb, rng, speed=None):
    # Standards (or speed-matched pseudo-standards) and oddballs, in time order.
    # Columns: start_time, stop_time, anchor_time (chunk centre), anchor, condition, speed.
    table = find_table(nwb)
    if table is None:
        raise KeyError(f"no table matching {CTX['table_regex']!r}: {list(nwb.intervals.keys())}")
    df = (nwb.intervals[table].to_dataframe()
          .sort_values("start_time").reset_index(drop=True))
    types = df[trial_type_col(df)].astype(str).to_numpy()
    t0_all, t1_all = df["start_time"].to_numpy(), df["stop_time"].to_numpy()
    def spd_all(idx):
        return window_speed(speed, t0_all[idx], t1_all[idx])

    cls, anchor, D0, G0 = oddball_classes(df, types)
    is_odd = np.array([c is not None for c in cls])

    # Oddballs: N_PER_ODDBALL of each class (after the optional running filter).
    parts = []
    for odd in CTX["oddballs"] or order_oddballs(cls[is_odd]):
        idx = np.flatnonzero(cls == odd)
        if MIN_SPEED is not None:
            idx = idx[spd_all(idx) >= MIN_SPEED]
        if len(idx) > N_PER_ODDBALL:
            idx = np.sort(rng.choice(idx, size=N_PER_ODDBALL, replace=False))
        if len(idx) == 0:
            continue
        kind = anchor[idx[0]]
        parts.append(pd.DataFrame({
            "start_time": t0_all[idx], "stop_time": t1_all[idx],
            "anchor_time": anchor_times(df, idx, kind, D0, G0, True), "anchor": kind,
            "condition": odd, "speed": spd_all(idx)}))
    if not parts:
        raise ValueError(f"no oddball trials found in '{table}'. Expected labels "
                         f"{CTX['oddballs']}, found {sorted(set(types))}")
    odd = pd.concat(parts, ignore_index=True)

    if CTX["standards_from"] == "presentations":
        # Presentations since the last oddball (large before the first one).
        since, last = np.empty(len(df), dtype=int), -10**9
        for i in range(len(df)):
            since[i] = i - last
            if is_odd[i]:
                last = i
        ok = standard_mask(df, types) & (since > CTX.get("skip_after", SKIP_AFTER_DEVIANT))
        std_idx = np.flatnonzero(ok)
        if MIN_SPEED is not None:
            std_idx = std_idx[spd_all(std_idx) >= MIN_SPEED]
        if len(std_idx) > MAX_STANDARDS:
            std_idx = np.sort(rng.choice(std_idx, size=MAX_STANDARDS, replace=False))
        # One copy of the standards per anchor the oddballs use, centred on the same event.
        std = []
        for kind in sorted(set(odd["anchor"])):
            idx = std_idx
            if kind == "next_onset":                     # next presentation must be a standard
                nxt_std = np.r_[types[1:] == STANDARD, False]
                idx = idx[nxt_std[idx]]
            if kind == "onset":                          # ...or the previous one, for "onset"
                prev_std = np.r_[False, types[:-1] == STANDARD]
                idx = idx[prev_std[idx]]
            std.append(pd.DataFrame({
                "start_time": t0_all[idx], "stop_time": t1_all[idx],
                "anchor_time": anchor_times(df, idx, kind, D0, G0, False), "anchor": kind,
                "condition": STANDARD, "speed": spd_all(idx)}))
        std = pd.concat(std, ignore_index=True)
    else:
        std = pseudo_standards(t0_all, t1_all, is_odd, odd, speed, rng)

    out = pd.concat([std, odd]).dropna(subset=["anchor_time"])
    return out.sort_values("anchor_time").reset_index(drop=True)


def pseudo_standards(t0_all, t1_all, is_odd, odd, speed, rng):
    # Random windows in the block, clear of every oddball, matched to the oddballs on speed.
    dur = float(np.median(odd["stop_time"] - odd["start_time"]))
    half = CHUNK_DUR / 2
    lo, hi = t0_all.min() + half, t1_all.max() - half
    c = rng.uniform(lo, hi, PSEUDO_CANDIDATES)

    # No overlap between the 1 s chunk and [oddball - EXCLUDE_BEFORE, oddball + EXCLUDE_AFTER].
    s = t0_all[is_odd] - EXCLUDE_BEFORE
    e = t1_all[is_odd] + EXCLUDE_AFTER
    clear = ~((c[:, None] + half > s[None, :]) & (c[:, None] - half < e[None, :])).any(1)
    c = c[clear]

    c_speed = window_speed(speed, c - dur / 2, c + dur / 2)
    if MIN_SPEED is not None:
        keep = c_speed >= MIN_SPEED
        c, c_speed = c[keep], c_speed[keep]

    o_speed = odd["speed"].to_numpy()
    if CTX["match_running"] and np.isfinite(o_speed).any() and np.isfinite(c_speed).any():
        pick = match_by_speed(c_speed, o_speed, MAX_STANDARDS, rng)
    else:
        if CTX["match_running"]:
            print("  ! no running speed found — pseudo-standards are NOT speed-matched")
        pick = rng.choice(len(c), size=min(MAX_STANDARDS, len(c)), replace=False)
    c = c[pick]
    return pd.DataFrame({"start_time": c - dur / 2, "stop_time": c + dur / 2,
                         "anchor_time": c, "anchor": "centre",
                         "condition": STANDARD, "speed": c_speed[pick]})


def read_batches(onsets):
    # Group sorted onsets into reads no longer than MAX_READ_SPAN_S, split at long gaps.
    batches, start = [], 0
    for i in range(1, len(onsets) + 1):
        if (i == len(onsets) or onsets[i] - onsets[start] > MAX_READ_SPAN_S
                or onsets[i] - onsets[i - 1] > MAX_READ_GAP_S):
            batches.append((start, i))
            start = i
    return batches

In [ ]:
#@title PSD of stimulus-centred chunks (Gaussian taper)
def series_time_base(es):
    if es.timestamps is not None:
        ts = np.asarray(es.timestamps[:])
        return ts, float(1.0 / np.median(np.diff(ts)))
    fs = float(es.rate)
    return es.starting_time + np.arange(es.data.shape[0]) / fs, fs


def log_bin_weights(f):
    # FFT bins (rows) -> log-spaced bins (columns), each column averaging its FFT bins.
    which = np.digitize(f, FREQ_EDGES) - 1
    W = np.zeros((len(f), N_FREQ_BINS))
    inside = (which >= 0) & (which < N_FREQ_BINS)
    W[np.flatnonzero(inside), which[inside]] = 1.0
    counts = W.sum(0)
    if (counts == 0).any():
        raise ValueError("a log bin has no FFT points; increase NFFT")
    return W / counts


def gaussian_taper(n_samp, fs, fwhm):
    # Gaussian centred on the chunk; its FWHM (in s) is the stimulus duration.
    t = (np.arange(n_samp) - (n_samp - 1) / 2) / fs
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return np.exp(-0.5 * (t / sigma) ** 2)


def chunk_log_psd(chunks, fs, taper):
    # Log10 power in log-spaced bins for chunks (channels x trials x samples).
    # Tapered, mean removed, zero-padded to NFFT, power averaged within each bin.
    # Channels are done one at a time to keep the zero-padded spectra small in memory.
    out = np.full(chunks.shape[:2] + (N_FREQ_BINS,), np.nan, np.float32)
    W = None
    for c in range(chunks.shape[0]):
        f, p = periodogram(chunks[c], fs=fs, window=taper, detrend="constant",
                           nfft=NFFT, axis=-1)
        if W is None:
            W = log_bin_weights(f)
        out[c] = np.log10(p @ W + np.finfo(float).tiny)
    return FREQ_CENTERS, out


def psd_for_channels(nwb, chans, centres, fwhm):
    # Log PSDs (channels x chunks x freqs) of 1 s chunks centred on `centres` (sorted),
    # for the channels in `chans` (columns series, col), in the order of its rows.
    chans = chans.reset_index(drop=True)
    onsets = np.asarray(centres, dtype=float) - CHUNK_DUR / 2     # chunk starts
    freqs, logpsd = None, None
    for name, ch_rows in chans.groupby("series"):
        es = lfp_series(nwb)[name]
        ts, fs = series_time_base(es)
        n_samp = int(round(CHUNK_DUR * fs))
        cols = np.sort(ch_rows["col"].to_numpy().astype(int))    # h5py needs increasing indices
        out_rows = [chans.index[(chans["series"] == name) & (chans["col"] == c)][0]
                    for c in cols]
        scale = float(getattr(es, "conversion", 1.0) or 1.0)

        chunks = np.full((len(cols), len(onsets), n_samp), np.nan, dtype=np.float32)
        for b0, b1 in read_batches(onsets):
            i0 = np.searchsorted(ts, onsets[b0])
            i1 = min(np.searchsorted(ts, onsets[b1 - 1]) + n_samp, len(ts))
            block = np.asarray(es.data[i0:i1, cols], dtype=np.float32) * scale
            for k in range(b0, b1):
                j = np.searchsorted(ts, onsets[k]) - i0
                if j >= 0 and j + n_samp <= block.shape[0]:
                    chunks[:, k, :] = block[j:j + n_samp].T
            del block

        f, lp = chunk_log_psd(chunks, fs, gaussian_taper(n_samp, fs, fwhm))
        if freqs is None:
            freqs = f
            logpsd = np.full((len(chans), len(onsets), len(freqs)), np.nan, np.float32)
        elif len(f) != len(freqs) or not np.allclose(f, freqs):
            lp = interp1d(f, lp, axis=-1, bounds_error=False)(freqs)   # probes at another fs
        logpsd[out_rows] = lp
    return freqs, logpsd


def extract_session(path):
    # Trial-by-channel log PSDs for one animal. Returns a dict ready for np.savez.
    subject = subject_of(path)
    rng = np.random.default_rng([RNG_SEED, int(subject)])
    io, nwb = stream_nwb(path)
    try:
        chans = sample_channels(lfp_channels(nwb), MAX_CH_PER_AREA, rng)
        if chans.empty:
            raise RuntimeError("no LFP channels in any main area")
        speed = running_speed(nwb)
        trials = build_trials(nwb, rng, speed)
        centres = trials["anchor_time"].to_numpy()
        is_std = trials["condition"].to_numpy() == STANDARD
        fwhm = TAPER_FWHM or float(np.median(trials["stop_time"].to_numpy()[is_std]
                                             - trials["start_time"].to_numpy()[is_std]))
        freqs, logpsd = psd_for_channels(nwb, chans, centres, fwhm)

        return {
            "subject": subject, "session": path, "context": CONTEXT,
            "psd_version": CTX.get("psd_version", 1),
            "freqs": freqs, "logpsd": logpsd,
            "condition": trials["condition"].to_numpy().astype(str),
            "centre": centres, "anchor": trials["anchor"].to_numpy().astype(str),
            "speed": trials["speed"].to_numpy(dtype=float),
            "taper_fwhm": fwhm,
            "electrode_id": chans["electrode_id"].to_numpy(),
            "location": chans["location"].to_numpy().astype(str),
            "area": chans["area"].to_numpy().astype(str),
            "series": chans["series"].to_numpy().astype(str),
        }
    finally:
        io.close()


def psd_file(subject):
    return os.path.join(PSD_DIR, f"{subject}_psd.npz")


def psd_up_to_date(subject):
    # Saved and made with the current trial classification for this context.
    f = psd_file(subject)
    if not os.path.exists(f):
        return False
    with np.load(f) as z:
        v = int(z["psd_version"]) if "psd_version" in z.files else 1
    return v >= CTX.get("psd_version", 1)


def load_psd(subject, psd_dir=None):
    with np.load(os.path.join(psd_dir or PSD_DIR, f"{subject}_psd.npz"), allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

In [ ]:
#@title Matched controls and V1 layers (stage 1b helpers)
EXTRA_VERSION = 1


def read_table(nwb, name):
    # Interval table sorted by time, without the bulky references to stimulus time series.
    t = nwb.intervals[name]
    try:
        df = t.to_dataframe(exclude={"timeseries"})
    except TypeError:
        df = t.to_dataframe()
    return df.sort_values("start_time").reset_index(drop=True)


def column_like(df, regex):
    return next((c for c in df.columns if re.fullmatch(regex, c)), None)


def control_table(nwb, spec):
    # Rows of this context's control block, identified by BlockType; tables whose name
    # matches spec["table_regex"] are checked first. Returns (rows, table names used).
    names = [n for n in nwb.intervals.keys() if re.search(r"(?i)control", n)]
    by_name = [n for n in names if re.search(spec["table_regex"], n)]
    parts, used = [], []
    for group in (by_name, [n for n in names if n not in by_name]):
        for n in group:
            df = read_table(nwb, n)
            bt = column_like(df, r"(?i)block_?type")
            if bt is not None:
                df = df[df[bt].astype(str).str.contains(spec["block_type"], regex=True)]
            elif group is not by_name:
                continue                         # without a BlockType column, trust names only
            if len(df):
                parts.append(df)
                used.append(n)
        if parts:
            break
    if not parts:
        return None, used
    return pd.concat(parts).sort_values("start_time").reset_index(drop=True), used


def control_trials(nwb, spec, rng):
    # Control presentations of every oddball stimulus and of the standard: same trial type
    # and, for gratings, exactly the orientation (and drift direction) it has in the
    # oddball block. Returns (DataFrame with centre and condition, table names used).
    ctrl, used = control_table(nwb, spec)
    if ctrl is None:
        return None, used
    main = read_table(nwb, find_table(nwb))
    m_types = main[trial_type_col(main)].astype(str).to_numpy()
    c_types = ctrl[trial_type_col(ctrl)].astype(str).to_numpy()
    om, oc = column_like(main, r"(?i)orientation"), column_like(ctrl, r"(?i)orientation")
    c_ori = pd.to_numeric(ctrl[oc], errors="coerce").to_numpy() if oc else None
    t0, t1 = ctrl["start_time"].to_numpy(), ctrl["stop_time"].to_numpy()
    parts = []
    targets = list(ODDBALLS or []) + ([STANDARD] if spec.get("include_standard", True) else [])
    for target in targets:
        ctype = control_type(target, spec.get("same_type", False))
        sel = c_types == ctype
        if ctype == "single" and om and oc:
            rows = standard_mask(main, m_types) if target == STANDARD else (m_types == target)
            ori = pd.to_numeric(main.loc[rows, om], errors="coerce").dropna().round(4)
            if ori.empty:
                continue
            sel &= np.isclose(c_ori, ori.mode()[0], atol=1e-3)
        idx = np.flatnonzero(sel)
        if len(idx) > MAX_CONTROL_PER_TYPE:
            idx = np.sort(rng.choice(idx, size=MAX_CONTROL_PER_TYPE, replace=False))
        if len(idx):
            parts.append(pd.DataFrame({"centre": 0.5 * (t0[idx] + t1[idx]), "condition": target}))
    if not parts:
        return None, used
    return pd.concat(parts).sort_values("centre").reset_index(drop=True), used


def needs_extra():
    return CONTEXT in LAYER_CONTEXTS or CONTEXT in CONTROLS


def extra_file(subject, psd_dir=None):
    return os.path.join(psd_dir or PSD_DIR, f"{subject}_extra_psd.npz")


def extra_up_to_date(subject):
    f = extra_file(subject)
    if not os.path.exists(f) or os.path.getmtime(f) < os.path.getmtime(psd_file(subject)):
        return False
    with np.load(f) as z:
        if int(z["extra_version"]) < EXTRA_VERSION:
            return False
        # A context that gained a control block after its file was made needs that part.
        return CONTEXT not in CONTROLS or "ctrl_tables" in z.files


def load_extra(subject, psd_dir=None):
    f = extra_file(subject, psd_dir)
    if not os.path.exists(f):
        return None
    with np.load(f, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def extract_extra(path):
    # Stage 1b for one animal, on top of its saved stage-1 file (which is not changed):
    #  - V1 top-up: channels from every V1 layer, up to MAX_CH_PER_LAYER counting the ones
    #    already sampled, at the same trials and with the same taper;
    #  - matched controls: the control-block presentations of each stimulus, on the main
    #    channels and the top-up channels.
    subject = subject_of(path)
    main = load_psd(subject)
    previous = load_extra(subject)                 # reuse its top-up spectra if they match
    rng = np.random.default_rng([RNG_SEED, int(subject), 2])
    fwhm = float(main["taper_fwhm"])
    io, nwb = stream_nwb(path)
    try:
        allch = lfp_channels(nwb)
        out = {"subject": subject, "extra_version": EXTRA_VERSION}

        top = allch.iloc[:0]
        if CONTEXT in LAYER_CONTEXTS:
            v1 = allch[allch["area"] == "V1"].copy()
            v1["layer"] = v1["location"].map(layer_of)
            have = [layer_of(loc) for loc, a in zip(main["location"], main["area"], strict=True)
                    if a == "V1"]
            taken = set(main["electrode_id"].astype(int))
            picked = []
            for L in LAYER_ORDER:
                need = MAX_CH_PER_LAYER - have.count(L)
                cand = v1[(v1["layer"] == L) & ~v1["electrode_id"].isin(taken)]
                if need > 0 and len(cand):
                    picked.append(cand.sample(n=min(need, len(cand)),
                                              random_state=int(rng.integers(2**31))))
            if picked:
                top = pd.concat(picked).reset_index(drop=True)
        out.update({"top_electrode_id": top["electrode_id"].to_numpy(dtype=int),
                    "top_location": top["location"].to_numpy().astype(str),
                    "top_series": top["series"].to_numpy().astype(str)})
        reuse = (previous is not None and "top_logpsd" in previous
                 and np.array_equal(previous["top_electrode_id"], out["top_electrode_id"]))
        if reuse:
            out["top_logpsd"] = previous["top_logpsd"]          # same channels, same trials
        elif len(top):
            _, out["top_logpsd"] = psd_for_channels(nwb, top, main["centre"], fwhm)

        spec = CONTROLS.get(CONTEXT)
        if spec:
            ctrl, used = control_trials(nwb, spec, rng)
            out["ctrl_tables"] = np.array(used, dtype=str)
            if ctrl is not None:
                main_ch = pd.DataFrame({"electrode_id": main["electrode_id"].astype(int),
                                        "series": main["series"].astype(str)})
                main_ch = main_ch.merge(allch[["electrode_id", "series", "col"]],
                                        on=["electrode_id", "series"], how="left")
                both = pd.concat([main_ch[["series", "col"]], top[["series", "col"]]],
                                 ignore_index=True)
                _, lp = psd_for_channels(nwb, both, ctrl["centre"].to_numpy(), fwhm)
                out["ctrl_logpsd"] = lp[:len(main_ch)]
                out["ctrl_top_logpsd"] = lp[len(main_ch):]
                out["ctrl_condition"] = ctrl["condition"].to_numpy().astype(str)
                out["ctrl_centre"] = ctrl["centre"].to_numpy()
        return out
    finally:
        io.close()

## 2 · Sessions and a check before running all

The first cell scans the other NWB files of the 15 animals once (metadata only, a few
minutes) and saves the list of tables in each file to the results folder. Each context's sessions are
derived from that list, and the cell prints every mismatch/oddball block it found, so if a
context gets no sessions you can see the real block name and adjust its pattern.

The second cell inspects the first session of each context: which LFP locations map to which main area
(anything *unmapped* is ignored; add it to `MAIN_AREAS` if it matters), the raw trial types
in the block, the running-speed series it found, and the trials that will be kept.

In [ ]:
#@title Find each context's sessions (scans once, then reads the saved inventory)
FORCE_RESCAN = False   #@param {type:"boolean"}

if FORCE_RESCAN or load_inventory() is None:
    inv = {}
    _all_paths = list_nwb_assets()
    for subj in [subject_of(p) for p in STANDARD_SESSION_PATHS]:
        cands = [p for p in _all_paths if p.startswith(f"sub-{subj}/")
                 and p not in STANDARD_SESSION_PATHS]
        for p in cands:
            try:
                io, nwb = stream_nwb(p)
                try:
                    inv[p] = sorted(nwb.intervals.keys())
                finally:
                    io.close()
            except Exception as e:
                print(f"  could not open {p}: {type(e).__name__}: {e}")
        print(f"{subj}: {len(cands)} other file(s) scanned")
    with open(INVENTORY_FILE, "w") as f:
        json.dump(inv, f, indent=1)

inv = load_inventory()
# Every oddball-type block found in the scanned files, so a naming mismatch is easy to spot.
blocks = pd.Series([t for tabs in inv.values() for t in tabs
                    if re.search(r"(?i)(mismatch|oddball)", t)]).value_counts()
print("Mismatch/oddball blocks in the scanned files (name: number of files):")
print(blocks.to_string() if len(blocks) else "  none")
print()
for c in RUN_CONTEXTS:
    sess = load_sessions(c)
    print(f"{c:>13}: {len(sess)} sessions" + ("" if sess else
          f"   <-- NONE: no block matches {CONTEXTS[c]['table_regex']!r}; "
          "edit that pattern in the configuration cell to one of the names above"))
    if c != "standard":
        with open(os.path.join(RESULTS_DIR, f"sessions_{c}.json"), "w") as f:
            json.dump(sess, f, indent=1)                     # record of what was used

In [ ]:
#@title Inspect the first session of each context
for ctx in RUN_CONTEXTS:
    if not set_context(ctx):
        continue
    banner(ctx)
    io, nwb = stream_nwb(SESSION_PATHS[0])
    try:
        _chans = lfp_channels(nwb)
        _table = find_table(nwb)
        _raw = nwb.intervals[_table].to_dataframe()
        _speed = running_speed(nwb)
        _ts_names = list(timeseries_in(nwb))
        _trials = build_trials(nwb, np.random.default_rng(0), _speed)
        _ctrl, _used = (control_trials(nwb, CONTROLS[ctx], np.random.default_rng(0))
                        if ctx in CONTROLS else (None, []))
    finally:
        io.close()

    print(f"Session: {SESSION_PATHS[0]}")
    print("LFP channels per main area:",
          _chans["area"].value_counts().reindex(AREA_ORDER).dropna().astype(int).to_dict())
    print("Unmapped locations:", sorted(_chans.loc[_chans["area"].isna(), "location"].unique()))
    print(f"Table '{_table}', raw trial types:",
          _raw[trial_type_col(_raw)].value_counts().to_dict())
    print("Time series in the file:", _ts_names)
    print("Running speed used:", "none found" if _speed is None
          else f"{len(_speed[1])} samples, median {np.nanmedian(_speed[1]):.2f}")
    print("Trials kept, chunk anchor and median running speed:")
    print(_trials.groupby(["condition", "anchor"])["speed"].agg(["size", "median"]).to_string())
    _v1 = _chans.loc[_chans["area"] == "V1", "location"].map(layer_of).value_counts()
    print("V1 LFP channels per layer:", {L: int(_v1.get(L, 0)) for L in LAYER_ORDER})
    if ctx in CONTROLS:
        print(f"Matched control from {_used or 'no table'}:",
              "NONE FOUND — check CONTROLS in the configuration cell" if _ctrl is None
              else _ctrl["condition"].value_counts().to_dict())

## 3 · Stage 1 — trial PSDs, one animal at a time

Context by context, each animal is streamed, chunked, and its PSDs saved before
the next one starts. Animals whose file is already saved are skipped, so re-running
resumes where it stopped. Set `FORCE_RECOMPUTE_PSD` to redo everything.

In [ ]:
#@title Run extraction over all contexts and animals (saved, resumable)
FORCE_RECOMPUTE_PSD = False   #@param {type:"boolean"}

for ctx in RUN_CONTEXTS:
    if not set_context(ctx):
        continue
    banner(ctx)
    for i, path in enumerate(SESSION_PATHS, start=1):
        subject = subject_of(path)
        tag = f"[{i}/{len(SESSION_PATHS)}] {subject}"
        if psd_up_to_date(subject) and not FORCE_RECOMPUTE_PSD:
            print(f"{tag}: already saved, skipped")
            continue
        try:
            res = extract_session(path)
            np.savez_compressed(psd_file(subject), **res)
            n_ch = pd.Series(res["area"]).value_counts().reindex(AREA_ORDER).dropna()
            n_tr = pd.Series(res["condition"]).value_counts()
            line = (f"{tag}: {len(res['area'])} ch {({k: int(v) for k, v in n_ch.items()})} | "
                    f"trials {({k: int(v) for k, v in n_tr.items()})}")
            if np.isfinite(res["speed"]).any():
                spd = pd.Series(res["speed"]).groupby(res["condition"]).median()
                line += f" | median speed {({k: round(float(v), 1) for k, v in spd.items()})}"
            print(line)
        except Exception as e:
            print(f"{tag}: FAILED — {type(e).__name__}: {e}")

### Stage 1b — matched controls and V1 layers

A second, separate file per animal (`<subject>_extra_psd.npz`), so the stage-1 files and
everything computed from them stay exactly as they were:

- **Matched controls** (standard and sequence contexts): the same stimulus presented in
  the context's control block, where the order is random and there is no regularity to
  violate. For a 45° oddball the control trials are the 45° gratings of the control block
  (same orientation and drift direction, read from the `Orientation` column), for halt the
  control halts, for omission the control omissions, and for the standard the control
  presentations of the standard's orientation. Up to `MAX_CONTROL_PER_TYPE` per stimulus,
  same 1 s chunks and Gaussian taper, same channels.
- **No control for sensorimotor or duration.** The open-loop replay block (Control block 4)
  presents the sensorimotor events only ~8 times per type, too few for a control, and no
  control block has the duration paradigm's gap deviants.
- **V1 layers**: the main sample has 10 random V1 channels, so a given layer (often L4 or
  L1) can be missing in an animal. Stage 1b adds V1 channels until each layer has up to
  `MAX_CH_PER_LAYER`, at the same trials as stage 1.

This streams each session once more (roughly as long as stage 1 for that context). To save
time, shorten `LAYER_CONTEXTS`; the controls are still read for every context in `CONTROLS`.
If a context gains a control later, its saved files are completed in place: their V1 top-up
spectra are reused and only the control trials are read.

In [ ]:
#@title Run stage 1b over all contexts and animals (saved, resumable)
FORCE_RECOMPUTE_EXTRA = False   #@param {type:"boolean"}

for ctx in RUN_CONTEXTS:
    if not set_context(ctx) or not needs_extra():
        continue
    banner(ctx)
    for i, path in enumerate(SESSION_PATHS, start=1):
        subject = subject_of(path)
        tag = f"[{i}/{len(SESSION_PATHS)}] {subject}"
        if not os.path.exists(psd_file(subject)):
            print(f"{tag}: no PSD file yet — run stage 1")
            continue
        if extra_up_to_date(subject) and not FORCE_RECOMPUTE_EXTRA:
            print(f"{tag}: already saved, skipped")
            continue
        try:
            res = extract_extra(path)
            np.savez_compressed(extra_file(subject), **res)
            main = load_psd(subject)
            lay = [layer_of(loc) for loc, a in zip(main["location"], main["area"], strict=True)
                   if a == "V1"]
            lay += [layer_of(loc) for loc in res["top_location"]]
            line = f"{tag}: V1 channels per layer {({L: lay.count(L) for L in LAYER_ORDER})}"
            if ctx in CONTROLS:
                n_c = pd.Series(res.get("ctrl_condition", [])).value_counts()
                line += (f" | control trials {({k: int(v) for k, v in n_c.items()})}" if len(n_c)
                         else f" | NO control trials (tables {list(res['ctrl_tables'])})")
            print(line)
        except Exception as e:
            print(f"{tag}: FAILED — {type(e).__name__}: {e}")

## 4 · Stage 2 — mutual information per animal

One MI file per animal and context, holding every contrast: oddball vs standard over the
main areas (the original analysis; the values are unchanged), the same vs the matched
control (and standard vs control), and both again over the V1 layers. Each comes with Δ
power, the mean difference of log power between the two classes in dB, computed on the same
balanced draws as the MI. MI files from before these additions are recomputed once
(`MI_VERSION`); the trial PSDs are not.

In [ ]:
#@title Gaussian-copula MI (vectorised over frequencies and shuffles)
def copnorm(x):
    # Rank-transform each column to a standard normal (the Gaussian copula).
    return ndtri(rankdata(x, axis=0) / (x.shape[0] + 1))


def gcmi_binary(cx, labels):
    # MI in bits between each column of cx (n x F, copula-normalised) and a binary class.
    # labels: S x n boolean, one labelling per row (the true one plus shuffles).
    # Closed form for Gaussian variables: I = H(X) - sum_c p_c H(X | c), and the
    # 0.5*log(2*pi*e) constants cancel, leaving log-variances.
    n = cx.shape[0]
    y1 = labels.astype(float)
    y0 = 1.0 - y1
    n1, n0 = y1.sum(1, keepdims=True), y0.sum(1, keepdims=True)
    cx2 = cx ** 2
    v1 = (y1 @ cx2 - (y1 @ cx) ** 2 / n1) / (n1 - 1)
    v0 = (y0 @ cx2 - (y0 @ cx) ** 2 / n0) / (n0 - 1)
    h_marg = 0.5 * np.log(cx.var(axis=0, ddof=1))
    h_cond = (n1 * 0.5 * np.log(v1) + n0 * 0.5 * np.log(v0)) / n
    return (h_marg - h_cond) / np.log(2)


def matched_standards(s_std, s_odd, rng, k=5):
    # One standard per oddball, drawn at random among the k nearest in running speed
    # (no reuse), so each draw reproduces this oddball type's own speed distribution.
    free = np.ones(len(s_std), bool)
    out = []
    for s in rng.permutation(s_odd):
        d = np.where(free, np.abs(s_std - s), np.inf)
        nn = np.argsort(d)[:k]
        nn = nn[np.isfinite(d[nn])]
        j = rng.choice(nn)
        free[j] = False
        out.append(j)
    return np.array(out)


def mi_one_contrast(feat_std, feat_odd, rng, s_std=None, s_odd=None):
    # Shuffle-corrected MI (bits), permutation p, number of trials per class and Δ power
    # (dB, oddball minus reference, from the same balanced draws), per frequency.
    # feat_std is the reference class: the standards, or the matched control trials.
    # With speeds given (sensorimotor), standards are re-matched to this oddball type in
    # every draw; otherwise both classes are subsampled at random to equal size.
    match = (s_std is not None and np.isfinite(s_std).all() and np.isfinite(s_odd).all()
             and len(feat_std) >= len(feat_odd))
    n = len(feat_odd) if match else min(len(feat_std), len(feat_odd))
    obs = np.empty((N_DRAWS, feat_std.shape[1]))
    null = np.empty((N_DRAWS, N_SHUFFLE, feat_std.shape[1]))
    dpow = np.empty((N_DRAWS, feat_std.shape[1]))
    y = np.r_[np.zeros(n, bool), np.ones(n, bool)]
    for d in range(N_DRAWS):
        if match:
            s = feat_std[matched_standards(s_std, s_odd, rng)]
            o = feat_odd
        else:
            s = feat_std[rng.choice(len(feat_std), n, replace=False)]
            o = feat_odd[rng.choice(len(feat_odd), n, replace=False)]
        dpow[d] = o.mean(0) - s.mean(0)
        cx = copnorm(np.vstack([s, o]))
        labels = np.vstack([y] + [rng.permutation(y) for _ in range(N_SHUFFLE)])
        mi = gcmi_binary(cx, labels)
        obs[d], null[d] = mi[0], mi[1:]

    obs_m = obs.mean(0)
    null_m = null.mean(0)                      # shuffle k averaged over draws: same statistic
    p = (1 + (null_m >= obs_m).sum(0)) / (N_SHUFFLE + 1)
    return obs_m - null_m.mean(0), p, n, 10 * dpow.mean(0)      # log10 power -> dB

In [ ]:
#@title Run MI over all contexts and animals (saved, resumable)
FORCE_RECOMPUTE_MI = False   #@param {type:"boolean"}


def mi_file(subject, mi_dir=None):
    return os.path.join(mi_dir or MI_DIR, f"{subject}_{MI_TAG}.npz")


def mi_up_to_date(subject, ctx=None):
    # Saved with the current MI version and newer than the PSD files it was computed from.
    d = context_dirs(ctx or CONTEXT)
    f = mi_file(subject, d["mi"])
    if not os.path.exists(f):
        return False
    srcs = [s for s in (os.path.join(d["psd"], f"{subject}_psd.npz"), extra_file(subject, d["psd"]))
            if os.path.exists(s)]
    if srcs and os.path.getmtime(f) < max(os.path.getmtime(s) for s in srcs):
        return False
    with np.load(f) as z:
        return "mi_version" in z.files and int(z["mi_version"]) >= MI_VERSION


def oddballs_for_run():
    # Fixed per context, or (duration context) the union of the classes found across
    # animals, so every animal's MI file has the same layout.
    if ODDBALLS is not None:
        return list(ODDBALLS)
    names = set()
    for path in SESSION_PATHS:
        if os.path.exists(psd_file(subject_of(path))):
            with np.load(psd_file(subject_of(path))) as z:
                names |= set(z["condition"].astype(str)) - {STANDARD}
    return order_oddballs(names)


def unit_masks(kind, data, extra):
    # Units of a contrast and, per unit, (mask over stage-1 channels, mask over top-up channels).
    if kind == "areas":
        return AREA_ORDER, {u: (data["area"] == u, None) for u in AREA_ORDER}
    lay = np.array([layer_of(loc) if a == "V1" else None
                    for loc, a in zip(data["location"], data["area"], strict=True)], dtype=object)
    top = (np.array([layer_of(loc) for loc in extra["top_location"]], dtype=object)
           if extra is not None and "top_location" in extra else np.array([], dtype=object))
    return LAYER_ORDER, {u: (lay == u, top == u) for u in LAYER_ORDER}


def unit_feature(main_lp, top_lp, masks):
    # Trials x freqs: log power averaged over the unit's channels (None if it has none).
    m, t = masks
    parts = [main_lp[m]]
    if t is not None and top_lp is not None and len(t) and t.any():
        parts.append(top_lp[t])
    stack = np.concatenate(parts, axis=0)
    if not len(stack):
        return None, 0
    return np.nanmean(stack, axis=0), len(stack)


def contrast_mi(data, extra, kind, ref, rng):
    # MI and Δ power per (column, unit, frequency).
    #   kind: "areas" (main areas) or "layers" (V1 layers);
    #   ref:  "standard" (each oddball vs the standards) or "control" (each oddball, and the
    #         standard itself, vs the same stimulus in the control block).
    cond, freqs = data["condition"], data["freqs"]
    anchors = data["anchor"] if "anchor" in data else np.full(len(cond), "centre")
    speed = data.get("speed") if (CTX["match_running"] and ref == "standard") else None
    units, masks = unit_masks(kind, data, extra)
    top = extra.get("top_logpsd") if extra is not None else None
    if ref == "standard":
        columns = list(ODDBALLS_RUN)
    else:
        ctrl_cond = extra["ctrl_condition"].astype(str)
        columns = order_oddballs([o for o in ODDBALLS_RUN if o in set(ctrl_cond)]
                                 + ([STANDARD] if STANDARD in set(ctrl_cond) else []))
        ctop = extra.get("ctrl_top_logpsd")
    shape = (len(columns), len(units), len(freqs))
    mi, p, dpow = np.full(shape, np.nan), np.full(shape, np.nan), np.full(shape, np.nan)
    n_trials = np.zeros(shape[:2], int)
    n_ch = np.zeros(len(units), int)

    for u_i, u in enumerate(units):
        feat, n = unit_feature(data["logpsd"], top, masks[u])
        if feat is None:
            continue
        n_ch[u_i] = n
        ok = ~np.isnan(feat).any(1)
        if ref == "control":
            cfeat, _ = unit_feature(extra["ctrl_logpsd"], ctop, masks[u])
            c_ok = ~np.isnan(cfeat).any(1)
        for c_i, col in enumerate(columns):
            s_ref = s_tgt = None
            if ref == "standard":
                is_tgt = ok & (cond == col)
                if not is_tgt.any():
                    continue
                # Standards centred on the same kind of event as this oddball.
                a_kind = pd.Series(anchors[is_tgt]).mode()[0]
                is_ref = ok & (cond == STANDARD) & (anchors == a_kind)
                f_ref, f_tgt = feat[is_ref], feat[is_tgt]
                if speed is not None:
                    s_ref, s_tgt = speed[is_ref], speed[is_tgt]
            else:
                is_tgt = ok & (cond == col)
                if col == STANDARD:
                    is_tgt &= anchors == "centre"
                f_ref, f_tgt = cfeat[c_ok & (ctrl_cond == col)], feat[is_tgt]
            min_trials = (CONTROLS[CONTEXT].get("min_trials", MIN_TRIALS)
                          if ref == "control" else MIN_TRIALS)
            if min(len(f_ref), len(f_tgt)) < min_trials:
                continue
            mi[c_i, u_i], p[c_i, u_i], n_trials[c_i, u_i], dpow[c_i, u_i] = mi_one_contrast(
                f_ref, f_tgt, rng, s_ref, s_tgt)
    return {"mi": mi, "p": p, "dpow": dpow, "n_trials": n_trials, "n_channels": n_ch,
            "units": np.array(units), "oddballs": np.array(columns)}


# Saved under these key prefixes, one MI file per animal and context:
#   ""          main areas, oddball vs standard (the original analysis, same values)
#   "ctrl_"     main areas, vs matched control
#   "lay_"      V1 layers, oddball vs standard
#   "layctrl_"  V1 layers, vs matched control
CONTRASTS = [("", "areas", "standard", 1), ("ctrl_", "areas", "control", 3),
             ("lay_", "layers", "standard", 4), ("layctrl_", "layers", "control", 5)]


def animal_mi(data, extra=None):
    subject = str(data["subject"])
    out = {"subject": subject, "context": CONTEXT, "freqs": data["freqs"],
           "mi_version": MI_VERSION}
    for prefix, kind, ref, seed in CONTRASTS:
        if ref == "control" and (extra is None or "ctrl_logpsd" not in extra):
            continue
        rng = np.random.default_rng([RNG_SEED, int(subject), seed])
        for k, v in contrast_mi(data, extra, kind, ref, rng).items():
            out[prefix + k] = v
    out["areas"] = out["units"]
    return out


for ctx in RUN_CONTEXTS:
    if not set_context(ctx):
        continue
    banner(ctx)
    ODDBALLS_RUN = oddballs_for_run()
    print("Oddball classes:", ODDBALLS_RUN)
    for i, path in enumerate(SESSION_PATHS, start=1):
        subject = subject_of(path)
        tag = f"[{i}/{len(SESSION_PATHS)}] {subject}"
        if not os.path.exists(psd_file(subject)):
            print(f"{tag}: no PSD file yet — run stage 1")
            continue
        if mi_up_to_date(subject) and not FORCE_RECOMPUTE_MI:
            print(f"{tag}: already saved, skipped")
            continue
        res = animal_mi(load_psd(subject), load_extra(subject))
        np.savez_compressed(mi_file(subject), **res)
        done = [AREA_ORDER[a] for a in np.flatnonzero(res["n_channels"])]
        extra_kinds = [p.rstrip("_") for p, *_ in CONTRASTS[1:] if p + "mi" in res]
        print(f"{tag}: MI for {len(done)} areas {done}" + (f" + {extra_kinds}" if extra_kinds else ""))

## 5 · Group summary

Per area and frequency: mean shuffle-corrected MI across animals, and a one-sided Wilcoxon
signed-rank test of MI > 0 across animals, FDR-corrected (Benjamini–Hochberg) across all
area × frequency cells of each oddball, plus the mean Δ power. Areas recorded in fewer than
`MIN_ANIMALS_STATS` animals are shown but not tested. The same is done for every contrast:
`AGGS_ALL` (oddball vs standard), `AGGS_CTRL` (vs matched control), `AGGS_LAY` and
`AGGS_LAYCTRL` (V1 layers). All of them go into one table, `figures/mi_summary_*.csv`.

In [ ]:
#@title Aggregate per-animal MI, every context
def bh_fdr(pvals):
    # Benjamini-Hochberg q-values; NaNs pass through.
    q = np.full_like(pvals, np.nan, dtype=float)
    ok = ~np.isnan(pvals)
    pv = pvals[ok]
    if pv.size == 0:
        return q
    order = np.argsort(pv)
    ranked = pv[order] * pv.size / np.arange(1, pv.size + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(pv)
    out[order] = np.minimum(ranked, 1.0)
    q[ok] = out
    return q


def aggregate(ctx, prefix=""):
    # Group MI and Δ power for one context and one contrast (see CONTRASTS) from the
    # per-animal files; None if there are none. Files older than their PSDs are left out.
    files = sorted(glob.glob(os.path.join(context_dirs(ctx)["mi"], f"*_{MI_TAG}.npz")))
    per_animal = []
    for f in files:
        if not mi_up_to_date(os.path.basename(f).split("_")[0], ctx):
            continue
        with np.load(f) as z:
            if prefix + "mi" in z.files:
                per_animal.append({k[len(prefix):]: z[k] for k in z.files if k.startswith(prefix)}
                                  | {"subject": z["subject"], "freqs": z["freqs"]})
    if not per_animal:
        return None

    units = [str(u) for u in per_animal[0]["units"]]
    # Align animals by oddball name (classes can differ between animals in the duration context).
    oddballs = order_oddballs(o for r in per_animal for o in r["oddballs"].astype(str))
    shape = (len(per_animal), len(oddballs), len(units), len(per_animal[0]["freqs"]))
    mi_all, dpow_all = np.full(shape, np.nan), np.full(shape, np.nan)
    for k, r in enumerate(per_animal):                      # animals x oddballs x units x freqs
        for j, o in enumerate(r["oddballs"].astype(str)):
            mi_all[k, oddballs.index(o)] = r["mi"][j]
            dpow_all[k, oddballs.index(o)] = r["dpow"][j]
    n_ch_all = np.stack([r["n_channels"] for r in per_animal])
    mi_mean = np.nanmean(mi_all, axis=0)
    q = np.full(mi_mean.shape, np.nan)
    for o_i in range(len(oddballs)):
        p = np.full(mi_mean.shape[1:], np.nan)
        for a_i in range(len(units)):
            vals = mi_all[:, o_i, a_i, :]
            vals = vals[~np.isnan(vals).any(1)]
            if len(vals) < MIN_ANIMALS_STATS:
                continue
            for f_i in range(mi_all.shape[-1]):
                p[a_i, f_i] = wilcoxon(vals[:, f_i], alternative="greater").pvalue
        q[o_i] = bh_fdr(p)

    # Animals with MI in a unit (for the control contrast, also needs control trials).
    n_animals_area = (~np.isnan(mi_all[..., 0])).any(1).sum(0)
    return {
        "context": ctx, "label": CONTEXTS[ctx]["label"], "prefix": prefix,
        "oddballs": oddballs, "areas": units,
        "subjects": [str(r["subject"]) for r in per_animal],
        "freqs": per_animal[0]["freqs"], "mi_all": mi_all, "mi_mean": mi_mean, "q": q,
        "dpow_all": dpow_all, "dpow_mean": np.nanmean(dpow_all, axis=0),
        "n_ch_all": n_ch_all, "n_animals_area": n_animals_area,
        "n_animals_odd": (~np.isnan(mi_all[..., 0])).sum(0),   # oddballs x units
        "show_areas": [a for a, n in zip(units, n_animals_area, strict=True)
                       if n >= MIN_ANIMALS_SHOW],
    }


# One dict per contrast, keyed by context. AGGS_ALL is the main one (oddball vs standard).
AGGS_KIND = {"std": {}, "ctrl": {}, "lay_std": {}, "lay_ctrl": {}}
KIND_PREFIX = {"std": "", "ctrl": "ctrl_", "lay_std": "lay_", "lay_ctrl": "layctrl_"}
AGGS_ALL, AGGS_CTRL = AGGS_KIND["std"], AGGS_KIND["ctrl"]
AGGS_LAY, AGGS_LAYCTRL = AGGS_KIND["lay_std"], AGGS_KIND["lay_ctrl"]
rows = []
for ctx in RUN_CONTEXTS:
    set_context(ctx)
    banner(ctx)
    for kind, prefix in KIND_PREFIX.items():
        if "ctrl" in kind and ctx not in CONTROLS:
            continue                          # e.g. MI files made while a control was tried
        g = aggregate(ctx, prefix)
        if g is None:
            continue
        AGGS_KIND[kind][ctx] = g
        # Long-format table of every contrast, saved next to the figures.
        for o_i, odd in enumerate(g["oddballs"]):
            for a_i, a in enumerate(g["areas"]):
                vals = g["mi_all"][:, o_i, a_i, :]
                n = int((~np.isnan(vals[:, 0])).sum())
                if n == 0:
                    continue
                sem = (np.nanstd(vals, axis=0, ddof=1) / np.sqrt(n) if n > 1
                       else np.full(vals.shape[1], np.nan))
                for f_i, f in enumerate(g["freqs"]):
                    rows.append({"context": ctx, "contrast": kind, "oddball": odd, "unit": a,
                                 "freq_hz": f, "mi_bits": g["mi_mean"][o_i, a_i, f_i],
                                 "sem": sem[f_i], "dpow_db": g["dpow_mean"][o_i, a_i, f_i],
                                 "n_animals": n, "q_fdr": g["q"][o_i, a_i, f_i]})
    if ctx not in AGGS_ALL:
        print("No MI files yet — skipped.")
        continue
    AGG = AGGS_ALL[ctx]
    print(f"{len(AGG['subjects'])} animals, oddball classes {AGG['oddballs']}")
    print("Contrasts:", {k: len(v[ctx]["subjects"]) for k, v in AGGS_KIND.items() if ctx in v},
          "(animals)")
    print(pd.DataFrame({"animals": AGG["n_animals_area"],
                        "channels (median)": np.nanmedian(np.where(AGG["n_ch_all"] > 0,
                                                                   AGG["n_ch_all"], np.nan), 0)},
                       index=AGG["areas"]).query("animals > 0").to_string())
    if ctx in AGGS_LAY:
        L = AGGS_LAY[ctx]
        n_med = np.nanmedian(np.where(L["n_ch_all"] > 0, L["n_ch_all"], np.nan), 0).round(1)
        print("V1 layers, animals:",
              dict(zip(L["areas"], L["n_animals_area"].tolist(), strict=True)),
              "| channels (median):", dict(zip(L["areas"], n_med.tolist(), strict=True)))
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
pd.DataFrame(rows).to_csv(os.path.join(RESULTS_DIR, "figures", f"mi_summary_{MI_TAG}.csv"),
                          index=False)

In [ ]:
#@title Running-speed check (should match across classes)
# Per animal, median running speed of the standards and of each oddball type. In the
# sensorimotor context the standards are matched to the oddballs, so these should agree;
# a large gap means a running confound is left in the MI.
for ctx, AGG in AGGS_ALL.items():
    set_context(ctx)
    rows = {}
    for subj in AGG["subjects"]:
        d = load_psd(subj)
        if "speed" in d and np.isfinite(d["speed"]).any():
            rows[subj] = pd.Series(d["speed"]).groupby(d["condition"]).median()
    banner(ctx)
    if not rows:
        print("No running speed stored for this context.")
        continue
    spd = pd.DataFrame(rows).T
    spd = spd.reindex(columns=[STANDARD] + order_oddballs(c for c in spd.columns if c != STANDARD))
    print(spd.round(1).to_string())
    print("Median across animals:", spd.median().round(1).to_dict())

## 6 · Publication figure

One figure per paradigm, built from what the earlier stages saved (nothing is recomputed),
with the layout of the manuscript's supplementary figure. All panels are V1 except the heatmaps:

- **A** · an example V1 trial for `PUB_EVENT[paradigm]`: LFP low-passed at `LOWPASS_HZ`,
  centred on the stimulus onset, above the Morlet spectrogram averaged over `EXAMPLE_N_AVG`
  trials of the same event on the same channel.
- **B** · V1 power spectrum of the standard and every event, mean ± SEM across animals.
- **C** · V1 spectra of the event and of the standard (solid) next to the same stimuli in the
  control block (dashed). Sensorimotor and duration have no usable control, so their panel C
  shows the single-trial power distributions behind the MI instead, and they have no E.
- **D** · every event vs the standard, all areas; **E** · vs the matched control. Each row in
  `PUB_HEATMAP_ROWS` is one heatmap row: `"dpow"` is the power difference (dB, red more and
  blue less power), `"mi"` the MI. Dots mark cells where the MI is significant across animals
  (FDR), in every row.

How the example is chosen, so that the effect is clear but typical:
1. **band**: half an octave around the peak of the group MI for V1 and that event;
2. **channel**: in each animal, the V1 channel (including the per-layer channels of stage 1b)
   with the largest effect in that band (Cohen's d of single-trial band power, event vs
   standard, in the direction of the group Δ power); its depth is printed in the title;
3. **animal**: the one at the `EXAMPLE_SUBJECT_PCTL` percentile of that effect across animals;
4. **trial**: among trials whose ±2 s window holds no other oddball, the one at the
   `EXAMPLE_TRIAL_PCTL` percentile of band power in the direction of the effect.

Everything is written to `publication/` in the results folder: `panels/` (about 20 PNGs,
`<paradigm>_<letter>_<name>.png`), one composite per paradigm
(`lfp_spectral_mi_<paradigm>.png/pdf`), the manuscript's static figure
(`supplementary-lfp-spectral-mi.png/pdf`, the `PUB_STATIC_CONTEXT` paradigm without a title),
and an index for section 7. Panels whose data did not change are not redrawn;
`FORCE_PANELS` redraws them and `FORCE_EXAMPLES` also re-reads the example traces.

In [ ]:
#@title Publication figure: V1 panels and one composite figure per paradigm
import pickle

from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy.fft import next_fast_len
from scipy.ndimage import gaussian_filter
from scipy.signal import butter, sosfiltfilt

FORCE_PANELS   = False   #@param {type:"boolean"}
FORCE_EXAMPLES = False   #@param {type:"boolean"}
EXAMPLE_BAND         = (20, 35)  # Hz; fallback band when an oddball carries no clear MI in an area
EXAMPLE_MIN_MI       = 0.005     # bits
EXAMPLE_SUBJECT_PCTL = 75        # animal at this percentile of the effect across animals
EXAMPLE_TRIAL_PCTL   = 80        # trial at this percentile of band power (effect direction)
EXAMPLE_N_AVG        = 15        # trials averaged in the example spectrogram (1 = single trial)
EXAMPLE_HALF_WINDOW  = 2.0       # s shown either side of the stimulus onset
LOWPASS_HZ           = 100       # example traces are low-passed here
PANEL_DPI            = 150

STANDARD_COLOR = "tab:blue"
ODDBALL_COLORS = {"orientation_45": "tab:red", "orientation_90": "tab:orange",
                  "halt": "tab:green", "omission": "grey"}
TIMING_COLORS = ["tab:purple", "tab:brown", "tab:pink", "tab:olive"]
CONTRAST_TEXT = {"std": ("Oddball vs standard", "oddball − standard"),
                 "ctrl": ("Vs matched control: the same stimulus in a random sequence",
                          "stimulus − same stimulus in control")}


def color_of(cond, oddballs):
    b = base_of(cond)
    if cond == STANDARD or b == "standard":
        return STANDARD_COLOR
    if b in ODDBALL_COLORS:
        return ODDBALL_COLORS[b]
    timing = [o for o in oddballs if re.match(r"(duration|isi)_", base_of(o))]
    return TIMING_COLORS[timing.index(cond) % len(TIMING_COLORS)] if cond in timing else "tab:cyan"


def plot_oddballs(agg):
    return [o for o in agg["oddballs"] if not o.endswith("_unclassified")]


def panel_dir(ctx):
    d = os.path.join(context_dirs(ctx)["figures"], "panels")
    os.makedirs(d, exist_ok=True)
    return d


def slug(x):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_")


def strip_spines(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def log_freq_axis(ax, axis="y"):
    getattr(ax, f"set_{axis}scale")("log")
    getattr(ax, f"set_{axis}ticks")([4, 8, 15, 30, 60, 100])
    getattr(ax, f"set_{axis}ticklabels")([4, 8, 15, 30, 60, 100])
    ax.minorticks_off()


def newest(pattern):
    files = glob.glob(pattern)
    return max(files, key=os.path.getmtime) if files else None


def stale(out, *sources):
    # Draw the panel if forced, missing, or older than the data it comes from.
    if FORCE_PANELS or not os.path.exists(out):
        return True
    src = [s for s in sources if s and os.path.exists(s)]
    return bool(src) and os.path.getmtime(out) < max(os.path.getmtime(s) for s in src)


def save_panel(fig, out):
    fig.savefig(out, dpi=PANEL_DPI, bbox_inches="tight")
    fig.savefig(out.replace(".png", ".pdf"), bbox_inches="tight")
    plt.close(fig)


def finite_or(v, default):
    return float(v) if np.isfinite(v) and v > 0 else default


def colour_limit(x, pct):
    # Upper colour limit: the pct-th percentile, but never below 60 % of the maximum, so a
    # strong effect confined to a few cells (e.g. one layer, one band) is not saturated away.
    x = np.abs(x[np.isfinite(x)])
    return finite_or(max(np.percentile(x, pct), 0.6 * x.max()), 0) if x.size else 0


# ── Heatmaps ------------------------------------------------------------------
GREY_RAMP = LinearSegmentedColormap.from_list("mi_grey", ["#ffffff", "#444444"])


def _cells(agg, show_areas, oddballs):
    show_areas = show_areas or agg["show_areas"]
    oddballs = oddballs or plot_oddballs(agg)
    o_idx = [agg["oddballs"].index(o) for o in oddballs]
    a_idx = [agg["areas"].index(a) for a in show_areas]
    return show_areas, oddballs, o_idx, a_idx


def _style_cell_axis(ax, show_areas, xlabels):
    ax.set_xticks(np.arange(len(show_areas)) + 0.5)
    ax.set_xticklabels(show_areas if xlabels else [], rotation=90, fontsize=8)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)


def _sig_dots(ax, agg, o_i, a_idx):
    # White dots with a dark rim stay visible on both the light and the darkest cells.
    xs, ys = np.nonzero(agg["q"][o_i][a_idx] < ALPHA)
    ax.scatter(xs + 0.5, agg["freqs"][ys], s=7, facecolor="white",
               edgecolor="#222222", linewidth=0.4, zorder=3)


def mi_heatmaps(agg, axes=None, show_areas=None, vmax=None, row_label=None, oddballs=None,
                xlabels=True):
    # One heatmap per oddball, each in its own colour (white -> oddball colour), all on one
    # MI scale. Returns (fig, axes, mappable) with a grey mappable for a shared colour bar.
    show_areas, oddballs, o_idx, a_idx = _cells(agg, show_areas, oddballs)
    all_odds = plot_oddballs(agg)
    data = agg["mi_mean"][o_idx][:, a_idx, :]
    vmax = vmax or colour_limit(data, 99.5) or 0.01
    x_edges = np.arange(len(a_idx) + 1)

    fig = None
    if axes is None:
        n = len(oddballs)
        fig, axes = plt.subplots(1, n, figsize=(2.9 * n + 1.4, 4.4), sharey=True,
                                 constrained_layout=True, squeeze=False)
        axes = axes[0]
    for k, (ax, odd) in enumerate(zip(axes, oddballs, strict=True)):
        col = color_of(odd, all_odds)
        cmap = LinearSegmentedColormap.from_list(f"mi_{k}", ["#ffffff", col])
        ax.set_facecolor("#e6e6e6")                      # grey = not recorded / not computed
        ax.pcolormesh(x_edges, FREQ_EDGES, data[k].T, cmap=cmap, vmin=0, vmax=vmax,
                      shading="flat", edgecolor="white", linewidth=0.3)
        _sig_dots(ax, agg, o_idx[k], a_idx)
        ax.set_title(label_of(odd), fontsize=11, color=col)
        _style_cell_axis(ax, show_areas, xlabels)
    log_freq_axis(axes[0])
    axes[0].set_ylabel(f"{row_label}\nFrequency (Hz)" if row_label else "Frequency (Hz)")
    sm = plt.cm.ScalarMappable(norm=Normalize(0, vmax), cmap=GREY_RAMP)
    return fig, axes, sm


def dpow_heatmaps(agg, axes, show_areas=None, vmax=None, row_label=None, oddballs=None):
    # Δ power (dB) per oddball on one symmetric diverging scale; red = more power.
    show_areas, oddballs, o_idx, a_idx = _cells(agg, show_areas, oddballs)
    data = agg["dpow_mean"][o_idx][:, a_idx, :]
    vmax = vmax or colour_limit(data, 99) or 1.0
    norm = Normalize(-vmax, vmax)
    x_edges = np.arange(len(a_idx) + 1)
    for k, ax in enumerate(axes):
        ax.set_facecolor("#e6e6e6")
        ax.pcolormesh(x_edges, FREQ_EDGES, data[k].T, cmap="RdBu_r", norm=norm,
                      shading="flat", edgecolor="white", linewidth=0.3)
        _sig_dots(ax, agg, o_idx[k], a_idx)
        _style_cell_axis(ax, show_areas, True)
    log_freq_axis(axes[0])
    axes[0].set_ylabel(f"{row_label}\nFrequency (Hz)" if row_label else "Frequency (Hz)")
    return plt.cm.ScalarMappable(norm=norm, cmap="RdBu_r")


def contrast_figure(agg, out, contrast, title=None, col_width=2.9, rows=("mi", "dpow")):
    # One column per oddball type; rows: MI ("mi") and/or Δ power ("dpow"), same cells.
    # Dots mark cells where the MI is significant across animals (FDR), in every row.
    odds = plot_oddballs(agg)
    n = len(odds)
    fig, axes = plt.subplots(len(rows), n, figsize=(col_width * n + 1.8, 4.1 * len(rows)),
                             sharey=True, constrained_layout=True, squeeze=False)
    for r, (kind, row_axes) in enumerate(zip(rows, axes, strict=True)):
        last = r == len(rows) - 1
        if kind == "mi":
            _, _, sm = mi_heatmaps(agg, axes=row_axes, row_label="Information", xlabels=last)
            label, extend = "MI (bits)", "max"
        else:
            sm = dpow_heatmaps(agg, row_axes, row_label="Δ power")
            label, extend = f"Δ power (dB)\n{CONTRAST_TEXT[contrast][1]}", "both"
            if r == 0:                                    # event titles on the first row
                for ax, odd in zip(row_axes, odds, strict=True):
                    ax.set_title(label_of(odd), fontsize=11, color=color_of(odd, odds))
            if not last:
                for ax in row_axes:
                    ax.set_xticklabels([])
        cb = fig.colorbar(sm, ax=row_axes, shrink=0.85, pad=0.01, extend=extend)
        cb.set_label(label)
        cb.outline.set_visible(False)
    fig.suptitle(title or CONTRAST_TEXT[contrast][0], fontsize=11)
    save_panel(fig, out)


# ── Per-animal area features, read once per context ----------------------------------
_FEATS = {}


def area_features(ctx, subjects):
    # Per animal: trial conditions/anchors and, per area, power in dB (trials x freqs)
    # averaged over the area's channels; the same for the control trials when present.
    key = (ctx, tuple(subjects))
    if key not in _FEATS:
        _FEATS.clear()
        set_context(ctx)
        out = {}
        for subj in subjects:
            d, e = load_psd(subj), load_extra(subj)
            has_ctrl = e is not None and "ctrl_logpsd" in e
            r = {"cond": d["condition"].astype(str),
                 "anchor": (d["anchor"] if "anchor" in d
                            else np.full(len(d["condition"]), "centre")).astype(str),
                 "ctrl_cond": e["ctrl_condition"].astype(str) if has_ctrl else None,
                 "feat": {}, "ctrl_feat": {}}
            for area in AREA_ORDER:
                sel = d["area"] == area
                if sel.any():
                    r["feat"][area] = 10 * np.nanmean(d["logpsd"][sel], axis=0)
                    if has_ctrl:
                        r["ctrl_feat"][area] = 10 * np.nanmean(e["ctrl_logpsd"][sel], axis=0)
            out[subj] = r
        _FEATS[key] = out
    return _FEATS[key]


def ref_mask(r, odd):
    # The standards an oddball is compared with: centred on the same kind of event.
    is_odd = r["cond"] == odd
    kind = pd.Series(r["anchor"][is_odd]).mode()[0] if is_odd.any() else "centre"
    return (r["cond"] == STANDARD) & (r["anchor"] == kind)


# ── Power spectra ---------------------------------------------------------------
def area_spectra(feats, area, conds):
    # Mean power spectrum (dB) per condition in one area: per animal over the area's
    # channels and trials, then mean and SEM across animals. Standards: the
    # stimulus-centred set when present.
    per = {c: [] for c in conds}
    for r in feats.values():
        if area not in r["feat"]:
            continue
        db, anchors, cond = r["feat"][area], r["anchor"], r["cond"]
        is_std = cond == STANDARD
        std_anchor = "centre" if (is_std & (anchors == "centre")).any() \
            else pd.Series(anchors[is_std]).mode()[0]
        for c in conds:
            sel = cond == c
            if c == STANDARD:
                sel &= anchors == std_anchor
            if sel.any():
                per[c].append(np.nanmean(db[sel], axis=0))
    return {c: np.array(v) for c, v in per.items() if len(v)}


def _mean_sem(v):
    m = v.mean(0)
    return m, (v.std(0, ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.zeros_like(m))


def psd_panel(feats, agg, area, out):
    odds = plot_oddballs(agg)
    spectra = area_spectra(feats, area, [STANDARD] + odds)
    if not spectra:
        return False
    fig, ax = plt.subplots(figsize=(3.8, 4.2), constrained_layout=True)
    for c, v in spectra.items():
        m, sem = _mean_sem(v)
        col = color_of(c, odds)
        ax.plot(agg["freqs"], m, color=col, lw=1.5,
                label="Standard" if c == STANDARD else label_of(c))
        ax.fill_between(agg["freqs"], m - sem, m + sem, color=col, alpha=0.15, lw=0)
    log_freq_axis(ax, "x")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB)")
    ax.set_title(f"{area}, n = {len(next(iter(spectra.values())))} animals", fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    strip_spines(ax)
    save_panel(fig, out)
    return True


def triad_panel(feats, agg, area, odd, out):
    # Standard, oddball, and each one's control-block counterpart (dashed).
    odds = plot_oddballs(agg)
    per = {k: [] for k in ("std", "std_ctrl", "odd", "odd_ctrl")}
    for r in feats.values():
        if area not in r["ctrl_feat"]:
            continue
        main, ctrl = r["feat"][area], r["ctrl_feat"][area]
        for k, c, is_ctrl in (("std", STANDARD, False), ("std_ctrl", STANDARD, True),
                              ("odd", odd, False), ("odd_ctrl", odd, True)):
            v = (ctrl[r["ctrl_cond"] == c] if is_ctrl
                 else main[(r["cond"] == c) & (r["anchor"] == "centre")])
            if len(v):
                per[k].append(np.nanmean(v, axis=0))
    if not per["odd"] or not per["odd_ctrl"]:
        return False
    col = color_of(odd, odds)
    fig, ax = plt.subplots(figsize=(3.8, 4.2), constrained_layout=True)
    for k, (c, ls, lab) in {"std": (STANDARD_COLOR, "-", "Standard"),
                            "std_ctrl": (STANDARD_COLOR, "--", "  same grating, control"),
                            "odd": (col, "-", label_of(odd)),
                            "odd_ctrl": (col, "--", "  same stimulus, control")}.items():
        if not per[k]:
            continue
        m, sem = _mean_sem(np.array(per[k]))
        ax.plot(agg["freqs"], m, color=c, ls=ls, lw=1.5, label=lab)
        ax.fill_between(agg["freqs"], m - sem, m + sem, color=c, alpha=0.12, lw=0)
    log_freq_axis(ax, "x")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (dB)")
    ax.set_title(f"{area}, n = {len(per['odd_ctrl'])} animals", fontsize=10)
    ax.legend(frameon=False, fontsize=7.5)
    strip_spines(ax)
    save_panel(fig, out)
    return True


# ── From power to information -------------------------------------------------------
def band_of(agg, area, odd):
    # Half an octave around the peak of the (lightly smoothed) group MI, its peak bin, and
    # whether the MI was large enough to use; otherwise EXAMPLE_BAND.
    o_i, a_i = agg["oddballs"].index(odd), agg["areas"].index(area)
    mi = pd.Series(agg["mi_mean"][o_i, a_i]).rolling(3, center=True, min_periods=1).mean()
    if not np.isfinite(mi).any() or np.nanmax(mi) < EXAMPLE_MIN_MI:
        lo, hi = EXAMPLE_BAND
        return (lo, hi), int(np.argmin(np.abs(agg["freqs"] - np.sqrt(lo * hi)))), False
    k = int(np.nanargmax(mi.to_numpy()))
    f = agg["freqs"][k]
    return (f / 2 ** 0.25, f * 2 ** 0.25), k, True


def band_mask(freqs, band, k):
    sel = (freqs >= band[0]) & (freqs <= band[1])
    sel[k] = True
    return sel


def dist_panel(feats, agg, area, odd, out, contrast):
    # Single-trial band power, pooled across animals after subtracting each animal's mean
    # of the reference trials (standards, or the same stimulus in the control block).
    odds = plot_oddballs(agg)
    band, k, _ = band_of(agg, area, odd)
    fsel = band_mask(agg["freqs"], band, k)
    vals = {"ref": [], "odd": [], "std": []}
    n_an = 0
    for r in feats.values():
        if area not in r["feat"] or (contrast == "ctrl" and area not in r["ctrl_feat"]):
            continue
        bp = r["feat"][area][:, fsel].mean(1)
        is_odd = (r["cond"] == odd) & np.isfinite(bp)
        if contrast == "std":
            ref = bp[ref_mask(r, odd) & np.isfinite(bp)]
        else:
            cbp = r["ctrl_feat"][area][:, fsel].mean(1)
            ref = cbp[(r["ctrl_cond"] == odd) & np.isfinite(cbp)]
            c_std = cbp[(r["ctrl_cond"] == STANDARD) & np.isfinite(cbp)]
            s = bp[(r["cond"] == STANDARD) & (r["anchor"] == "centre") & np.isfinite(bp)]
            if len(c_std) and len(s):
                vals["std"].append(s - c_std.mean())
        if not is_odd.any() or len(ref) < 2:
            continue
        vals["ref"].append(ref - ref.mean())
        vals["odd"].append(bp[is_odd] - ref.mean())
        n_an += 1
    if n_an == 0:
        return False
    vals = {key: np.concatenate(v) for key, v in vals.items() if v}
    pooled = np.concatenate(list(vals.values()))
    lo, hi = np.percentile(pooled, [0.5, 99.5])
    bins = np.linspace(lo, hi, 36)

    col = color_of(odd, odds)
    ref_lab = "Standard" if contrast == "std" else "Same stimulus, control block"
    ref_col = STANDARD_COLOR if contrast == "std" else "#555555"
    fig, ax = plt.subplots(figsize=(3.8, 4.2), constrained_layout=True)
    ax.hist(vals["ref"], bins=bins, density=True, color=ref_col, alpha=0.25, label=ref_lab)
    ax.hist(vals["ref"], bins=bins, density=True, histtype="step", color=ref_col, lw=1)
    ax.hist(vals["odd"], bins=bins, density=True, color=col, alpha=0.3, label=label_of(odd))
    ax.hist(vals["odd"], bins=bins, density=True, histtype="step", color=col, lw=1.4)
    if contrast == "ctrl" and "std" in vals:
        ax.hist(vals["std"], bins=bins, density=True, histtype="step", color=STANDARD_COLOR,
                lw=1.4, label="Standard (vs its control)")
    for v, c in ((vals["ref"], ref_col), (vals["odd"], col)):
        ax.axvline(np.mean(v), color=c, lw=1, ls="--")
    o_i, a_i = agg["oddballs"].index(odd), agg["areas"].index(area)
    txt = (f"MI = {agg['mi_mean'][o_i, a_i, k]:.3f} bits\n"
           f"Δ = {np.nanmean(agg['dpow_mean'][o_i, a_i, fsel]):+.2f} dB")
    if contrast == "ctrl" and STANDARD in agg["oddballs"]:
        s_i = agg["oddballs"].index(STANDARD)
        txt += (f"\nstandard vs control: {agg['mi_mean'][s_i, a_i, k]:.3f} bits, "
                f"{np.nanmean(agg['dpow_mean'][s_i, a_i, fsel]):+.2f} dB")
    ax.text(0.02, 0.98, txt, transform=ax.transAxes, va="top", fontsize=7.5)
    ax.set_xlabel(f"{band[0]:.0f}–{band[1]:.0f} Hz power,\nrelative to the "
                  f"{'standard' if contrast == 'std' else 'control'} mean (dB)")
    ax.set_ylabel("Trials (density)")
    ax.set_title(f"{area}: single trials, n = {n_an} animals", fontsize=10)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.3)                        # room for the text
    ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.27))
    strip_spines(ax)
    save_panel(fig, out)
    return True


# ── Example traces ----------------------------------------------------------------
def morlet_power(x, fs, freqs, n_cycles=6):
    # Power of x at each frequency, Morlet wavelets (FWHM = n_cycles / f), FFT convolution.
    t = np.arange(-2, 2, 1 / fs)
    half, n = len(t) // 2, len(x)
    nfft = next_fast_len(n + len(t) - 1)
    X = np.fft.fft(x, nfft)
    out = np.empty((len(freqs), n))
    for i, f in enumerate(freqs):
        w = np.exp(2j * np.pi * f * t) * np.exp(-4 * np.log(2) * t ** 2 / (n_cycles / f) ** 2)
        W = np.fft.fft(w, nfft)
        out[i] = np.abs(np.fft.ifft(X * W / np.abs(W).max())[half:half + n]) ** 2
    return out


def channel_candidates(d, e, area):
    # (trials x freqs log power, location, electrode id, series) of every channel of the
    # area in this animal; V1 also gets the per-layer channels of stage 1b.
    out = [(d["logpsd"][i], str(d["location"][i]), int(d["electrode_id"][i]), str(d["series"][i]))
           for i in np.flatnonzero(d["area"] == area)]
    if area == "V1" and e is not None and "top_logpsd" in e:
        out += [(e["top_logpsd"][i], str(e["top_location"][i]), int(e["top_electrode_id"][i]),
                 str(e["top_series"][i])) for i in range(len(e["top_location"]))]
    return out


def cohen_d(a, b):
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 3 or len(b) < 3:
        return np.nan
    sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                 / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sd if sd > 0 else np.nan


def plan_examples(ctx, agg, keys):
    # For every (area, oddball) in keys: the band, the effect direction, and the animals
    # ranked from the EXAMPLE_SUBJECT_PCTL one outwards, each with its best channel.
    set_context(ctx)
    data = {s: (load_psd(s), load_extra(s)) for s in agg["subjects"]}
    plans = {}
    for area, odd in keys:
        if area not in agg["areas"] or odd not in agg["oddballs"]:
            continue
        band, k, _ = band_of(agg, area, odd)
        fsel = band_mask(agg["freqs"], band, k)
        o_i, a_i = agg["oddballs"].index(odd), agg["areas"].index(area)
        dp = np.nanmean(agg["dpow_mean"][o_i, a_i, fsel])
        sign = -1.0 if np.isfinite(dp) and dp < 0 else 1.0
        ranked = []
        for s, (d, e) in data.items():
            cond = d["condition"].astype(str)
            anchors = (d["anchor"] if "anchor" in d else np.full(len(cond), "centre")).astype(str)
            is_odd = cond == odd
            if not is_odd.any():
                continue
            is_ref = ref_mask({"cond": cond, "anchor": anchors}, odd)
            best = None
            for lp, loc, eid, ser in channel_candidates(d, e, area):
                bp = lp[:, fsel].mean(1)
                eff = sign * cohen_d(bp[is_odd], bp[is_ref])
                if np.isfinite(eff) and (best is None or eff > best["d"]):
                    best = {"d": eff, "location": loc, "eid": eid, "series": ser, "bp": bp}
            if best:
                ranked.append((s, best))
        if not ranked:
            continue
        ranked.sort(key=lambda x: x[1]["d"])
        i = int(round(EXAMPLE_SUBJECT_PCTL / 100 * (len(ranked) - 1)))
        order = sorted(range(len(ranked)), key=lambda j: (abs(j - i), -j))
        plans[(area, odd)] = {"band": band, "sign": sign, "n_animals": len(ranked),
                              "candidates": [ranked[j] + (j,) for j in order]}
    return plans, data


def fetch_examples(subj, wanted, d, rng):
    # Read the example windows of one animal: {(area, odd): (candidate, plan, rank)} -> traces.
    path = next(p for p in SESSION_PATHS if subject_of(p) == subj)
    H, pad = EXAMPLE_HALF_WINDOW, EXAMPLE_HALF_WINDOW + 2.0           # pad: wavelet edges
    io, nwb = stream_nwb(path)
    try:
        chans = lfp_channels(nwb)
        table = (nwb.intervals[find_table(nwb)].to_dataframe()
                 .sort_values("start_time").reset_index(drop=True))
        types = table[trial_type_col(table)].astype(str).to_numpy()
        cls, anchor, D0, G0 = oddball_classes(table, types)
        t0, t1 = table["start_time"].to_numpy(), table["stop_time"].to_numpy()
        # Any non-standard event counts against the one-oddball-per-window rule.
        other = (types != STANDARD) & ~np.char.startswith(types.astype(str), "sequence_")
        def single(r):
            return int((other & (t0 < t0[r] + H) & (t1 > t0[r] - H)).sum()) == 1

        plans = {}
        for key, (cand, plan, _) in wanted.items():
            area, odd = key
            rows = np.flatnonzero(cls == odd)
            if not len(rows):
                continue
            ok_rows = np.array([single(r) for r in rows])
            if not ok_rows.any():
                continue
            row_anchor = anchor_times(table, rows, anchor[rows[0]], D0, G0, True)
            bp = cand["bp"]
            ks = np.flatnonzero((d["condition"] == odd) & np.isfinite(bp))
            pairs = []
            for k in ks:
                j = int(np.argmin(np.abs(row_anchor - d["centre"][k])))
                if ok_rows[j]:
                    pairs.append((k, rows[j]))
            if not pairs:
                continue
            vals = np.array([bp[k] for k, _ in pairs])
            pct = EXAMPLE_TRIAL_PCTL if plan["sign"] > 0 else 100 - EXAMPLE_TRIAL_PCTL
            k_pick, r_pick = pairs[int(np.argmin(np.abs(vals - np.percentile(vals, pct))))]
            pool = rows[ok_rows]
            avg = (pool if len(pool) <= EXAMPLE_N_AVG
                   else np.sort(rng.choice(pool, EXAMPLE_N_AVG, replace=False)))
            hit = chans[(chans["electrode_id"] == cand["eid"]) & (chans["series"] == cand["series"])]
            if hit.empty:
                continue
            plans[key] = {"k": k_pick, "row": r_pick, "avg": avg, "col": int(hit["col"].iloc[0])}

        # Every window is read once, with all the columns any example needs from it.
        need = {}
        for key, p in plans.items():
            ser = wanted[key][0]["series"]
            for r in {p["row"], *p["avg"].tolist()}:
                need.setdefault((ser, r), set()).add(p["col"])
        raw, tbase = {}, {}
        for (ser, r), cols in sorted(need.items()):
            es = lfp_series(nwb)[ser]
            if ser not in tbase:
                tbase[ser] = series_time_base(es)
            ts, fs = tbase[ser]
            cols = sorted(cols)
            scale = float(getattr(es, "conversion", 1.0) or 1.0) * 1e6           # microvolts
            i0, i1 = np.searchsorted(ts, [t0[r] - pad, t0[r] + pad])
            block = np.asarray(es.data[i0:i1, cols], dtype=float) * scale
            for j, c in enumerate(cols):
                raw[(ser, r, c)] = (ts[i0:i1] - t0[r], block[:, j])

        freqs_w = np.geomspace(FMIN, FMAX, 60)
        traces = {}
        for key, p in plans.items():
            cand, plan, rank = wanted[key]
            ser, c = cand["series"], p["col"]
            fs = tbase[ser][1]
            t, x = raw[(ser, p["row"], c)]
            if fs > 2.5 * LOWPASS_HZ:
                x = sosfiltfilt(butter(4, LOWPASS_HZ, fs=fs, output="sos"), x)
            keep = np.abs(t) <= H
            # Spectrogram averaged over trials of this type (same channel, same alignment).
            P_list, n_s, t_ref = [], None, None
            for r in p["avg"]:
                tt, xx = raw[(ser, r, c)]
                if len(xx) < 2 * pad * fs - 2:
                    continue                                   # window cut by the recording edge
                kk = np.abs(tt) <= H
                pw = morlet_power(xx - xx.mean(), fs, freqs_w)[:, kk]
                t_ref = tt[kk] if t_ref is None else t_ref
                n_s = pw.shape[1] if n_s is None else min(n_s, pw.shape[1])
                P_list.append(pw)
            if not P_list:
                continue
            P = np.mean([pw[:, :n_s] for pw in P_list], axis=0)
            step = max(1, int(fs // 250))                                  # ~250 Hz to draw
            avg_db = 10 * np.log10(P / np.median(P, axis=1, keepdims=True))
            avg_t = t_ref[:n_s]
            r = p["row"]
            boxes = []
            for j in np.flatnonzero((t1 > t0[r] - H) & (t0 < t0[r] + H)):
                if cls[j] is not None:
                    boxes.append((t0[j] - t0[r], t1[j] - t0[r], str(cls[j])))
                elif types[j] == STANDARD and CTX["standards_from"] == "presentations":
                    boxes.append((t0[j] - t0[r], t1[j] - t0[r], STANDARD))
            traces[key] = {
                "t": t[keep], "x": x[keep], "fs": fs, "boxes": boxes,
                "anchor_offset": float(d["centre"][p["k"]] - t0[r]),
                "anchor": str(d["anchor"][p["k"]]) if "anchor" in d else "centre",
                "subject": subj, "location": cand["location"], "n_oddballs": 1,
                "avg_t": avg_t[::step], "avg_db": avg_db[:, ::step], "freqs_w": freqs_w,
                "n_avg": len(P_list),
                "band": plan["band"], "d": cand["d"] * plan["sign"],
                "animal_rank": (rank + 1, plan["n_animals"])}
        return traces
    finally:
        io.close()


def example_cache(ctx):
    return os.path.join(panel_dir(ctx), "example_traces_v2.pkl")


def choose_examples(ctx, agg, keys):
    # {(area, oddball): trace} for the requested keys. Cached on disk (examples chosen in
    # earlier runs are reused); everything is re-chosen when the MI changes.
    cache = example_cache(ctx)
    mi_src = newest(os.path.join(context_dirs(ctx)["mi"], f"*_{MI_TAG}.npz"))
    traces = {}
    if (os.path.exists(cache) and not FORCE_EXAMPLES
            and not (mi_src and os.path.getmtime(cache) < os.path.getmtime(mi_src))):
        with open(cache, "rb") as f:
            traces = pickle.load(f)
    missing = [k for k in keys if k not in traces]
    if not missing:
        return traces
    plans, data = plan_examples(ctx, agg, missing)
    rng = np.random.default_rng(RNG_SEED)
    pos = {key: 0 for key in plans}                 # which ranked animal each example is on
    while pos:
        subj = plans[next(iter(pos))]["candidates"][pos[next(iter(pos))]][0]
        keys = [k for k, j in pos.items() if plans[k]["candidates"][j][0] == subj]
        wanted = {}
        for k in keys:
            _, cand, rank = plans[k]["candidates"][pos[k]]
            wanted[k] = (cand, plans[k], rank)
        got = fetch_examples(subj, wanted, data[subj][0], rng)
        for k in keys:
            if k in got:
                traces[k] = got[k]
                del pos[k]
            else:                                   # no usable window: next animal in rank
                pos[k] += 1
                if pos[k] >= len(plans[k]["candidates"]):
                    del pos[k]
    del data
    with open(cache, "wb") as f:
        pickle.dump(traces, f)
    return traces


def example_panel(agg, area, odd, tr, out):
    odds = plot_oddballs(agg)
    H = EXAMPLE_HALF_WINDOW
    col = color_of(odd, odds)
    db = gaussian_filter(tr["avg_db"], sigma=(1.0, 2.0))
    vmax = finite_or(np.percentile(np.abs(db), 99), 1.0)

    fig = plt.figure(figsize=(6.6, 4.4), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 0.03], height_ratios=[1, 1.2])
    ax_l = fig.add_subplot(gs[0, 0])
    ax_w = fig.add_subplot(gs[1, 0], sharex=ax_l)
    cax = fig.add_subplot(gs[1, 1])
    for a, b, cond in tr["boxes"]:
        ax_l.axvspan(max(a, -H), min(b, H), color=color_of(cond, odds),
                     alpha=0.15 if cond == STANDARD else 0.35, lw=0)
    ax_l.plot(tr["t"], tr["x"], color="k", lw=0.7)
    if tr["anchor"] != "centre" and abs(tr["anchor_offset"]) > 1e-3:
        for ax in (ax_l, ax_w):
            ax.axvline(tr["anchor_offset"], color=col, lw=1, ls="--")
    mesh = ax_w.pcolormesh(tr["avg_t"], tr["freqs_w"], db, cmap="RdBu_r", vmin=-vmax,
                           vmax=vmax, shading="gouraud", rasterized=True)
    for f in tr["band"]:
        ax_w.axhline(f, color="k", lw=0.6, ls=":")
    log_freq_axis(ax_w)
    ax_l.set_xlim(-H, H)
    ax_l.set_ylabel(f"LFP (µV, <{LOWPASS_HZ} Hz)")
    ax_l.tick_params(labelbottom=False)
    strip_spines(ax_l)
    ax_l.text(0.99, 0.98, f"single trial · {tr['band'][0]:.0f}–{tr['band'][1]:.0f} Hz power at "
              f"{EXAMPLE_TRIAL_PCTL}th pct · d = {tr['d']:+.2f}",
              transform=ax_l.transAxes, ha="right", va="top", fontsize=7, color="#555555")
    ax_w.set_ylabel("Frequency (Hz)")
    ax_w.set_xlabel("Time from stimulus onset (s)")
    ax_l.set_title(f"{label_of(odd)} · {area} ({tr['location']}) · animal {tr['subject']}",
                   fontsize=10, color=col)
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label(f"Power (dB vs median)\nmean of {tr['n_avg']} trials")
    cb.outline.set_visible(False)
    save_panel(fig, out)




# ── Publication figure ------------------------------------------------------------------
# One figure per paradigm, all in V1 except the heatmaps:
#   A  example trial + trial-averaged spectrogram for PUB_EVENT[ctx]
#   B  V1 power spectra, standard and every event
#   C  V1 spectra of PUB_EVENT and the standard next to the same stimuli in the control
#      block (dashed); without a control (duration): single-trial power distributions
#   D  heatmaps, every event vs the standard, all areas
#   E  heatmaps vs the matched control, all areas (contexts with a control block)
PUB_DIR = os.path.join(RESULTS_DIR, "publication")
PUB_PANELS = os.path.join(PUB_DIR, "panels")
PUB_MANIFEST = os.path.join(PUB_DIR, "publication_manifest.json")
PUB_AREA = "V1"
PUB_EVENT = {"standard": "orientation_45", "sequence": "halt",
             "sensorimotor": "motor_orientation_45", "duration": "isi_150ms"}
PUB_HEATMAP_ROWS = ("dpow",)        # ("mi", "dpow") adds the MI row above the Δ power row
PUB_STATIC_CONTEXT = "standard"     # the paradigm of the manuscript's static figure
PUB_FIG_WIDTH = 12.0                # inches; composites are saved at PUB_DPI
PUB_DPI = 200
CONTROL_NOTES = {
    "standard": "Control: Control block 1, the same gratings, halts and omissions in random "
                "order with the same timing.",
    "sequence": "Control: Control block 2, the sequence gratings shuffled.",
    "sensorimotor": "No control: the open-loop replay presents each event only ~8 times. C "
                    "shows single-trial power distributions instead, and there is no panel E.",
    "duration": "No control block has the same gap deviants: C shows single-trial power "
                "distributions instead, and there is no panel E.",
}
CONTROL_TITLES = {}
os.makedirs(PUB_PANELS, exist_ok=True)


def publication_panels(ctx, agg):
    # Render the panels of one paradigm (skipping up-to-date ones); {letter: path}.
    feats = area_features(ctx, agg["subjects"])
    odds = plot_oddballs(agg)
    event = PUB_EVENT.get(ctx) if PUB_EVENT.get(ctx) in odds else odds[0]
    ctrl = AGGS_CTRL.get(ctx)
    mi_src = newest(os.path.join(context_dirs(ctx)["mi"], f"*_{MI_TAG}.npz"))
    psd_src = newest(os.path.join(context_dirs(ctx)["psd"], "*.npz"))
    path = lambda name: os.path.join(PUB_PANELS, f"{ctx}_{name}.png")  # noqa: E731
    out = {}

    traces = choose_examples(ctx, agg, [(PUB_AREA, event)])
    if (PUB_AREA, event) in traces:
        f = path("A_example")
        if FORCE_EXAMPLES or stale(f, example_cache(ctx)):
            example_panel(agg, PUB_AREA, event, traces[(PUB_AREA, event)], f)
        out["A"] = f
    f = path("B_spectra")
    if not stale(f, psd_src) or psd_panel(feats, agg, PUB_AREA, f):
        out["B"] = f
    if ctrl is not None and event in ctrl["oddballs"]:
        f = path("C_spectra_control")
        if not stale(f, psd_src) or triad_panel(feats, agg, PUB_AREA, event, f):
            out["C"] = f
    else:
        f = path("C_single_trials")
        if not stale(f, mi_src, psd_src) or dist_panel(feats, agg, PUB_AREA, event, f, "std"):
            out["C"] = f
    f = path("D_heatmap_standard")
    if stale(f, mi_src):
        contrast_figure(agg, f, "std", rows=PUB_HEATMAP_ROWS)
    out["D"] = f
    if ctrl is not None:
        f = path("E_heatmap_control")
        if stale(f, mi_src):
            contrast_figure(ctrl, f, "ctrl", title=CONTROL_TITLES.get(ctx), rows=PUB_HEATMAP_ROWS)
        out["E"] = f
    return out, event


def compose_figure(panels, out_base, title=None):
    # Lay the panels out like the manuscript figure: A | B | C on top, then D and E at full
    # width, each at its own aspect ratio; letters at the top left. Writes PNG and PDF.
    from PIL import Image as _PIL
    ims = {k: np.asarray(_PIL.open(p).convert("RGB")) for k, p in panels.items()}
    W, gap, letter_h = PUB_FIG_WIDTH, 0.12, 0.25
    top = [k for k in "ABC" if k in ims]
    aspect = {k: ims[k].shape[1] / ims[k].shape[0] for k in ims}
    usable = W - gap * (len(top) - 1)
    row_h = usable / sum(aspect[k] for k in top)
    rows = [(top, row_h)] + [([k], W / aspect[k]) for k in "DE" if k in ims]
    title_h = 0.35 if title else 0.0
    H = title_h + sum(h + letter_h for _, h in rows) + gap * (len(rows) - 1)
    fig = plt.figure(figsize=(W, H))
    if title:
        fig.text(0.5, 1 - 0.12 / H, title, ha="center", va="top", fontsize=13)
    y = title_h
    for keys, h in rows:
        x = 0.0
        for k in keys:
            w = aspect[k] * h
            ax = fig.add_axes([x / W, 1 - (y + letter_h + h) / H, w / W, h / H])
            ax.imshow(ims[k], interpolation="lanczos")
            ax.axis("off")
            fig.text(x / W, 1 - y / H, k, ha="left", va="top", fontsize=20,
                     fontweight="bold")
            x += w + gap
        y += letter_h + h + gap
    for ext in ("png", "pdf"):
        fig.savefig(f"{out_base}.{ext}", dpi=PUB_DPI)
    plt.close(fig)
    return f"{out_base}.png"


manifest = {}
for ctx, AGG in AGGS_ALL.items():
    set_context(ctx)
    banner(ctx)
    panels, event = publication_panels(ctx, AGG)
    figure = compose_figure(panels, os.path.join(PUB_DIR, f"lfp_spectral_mi_{ctx}"),
                            title=f"{AGG['label']}: V1 example, {label_of(event)}")
    rel = lambda f: os.path.relpath(f, PUB_DIR).replace(os.sep, "/")  # noqa: E731
    manifest[ctx] = {"label": AGG["label"], "event": event, "event_label": label_of(event),
                     "n_animals": len(AGG["subjects"]), "note": CONTROL_NOTES.get(ctx, ""),
                     "panels": {k: rel(v) for k, v in panels.items()}, "figure": rel(figure)}
    print(f"panels {sorted(panels)} -> {figure}")

if PUB_STATIC_CONTEXT in manifest:                    # the manuscript's static figure
    static = compose_figure(publication_panels(PUB_STATIC_CONTEXT,
                                               AGGS_ALL[PUB_STATIC_CONTEXT])[0],
                            os.path.join(PUB_DIR, "supplementary-lfp-spectral-mi"))
    print(f"\nStatic figure: {static}")
with open(PUB_MANIFEST, "w") as f:
    json.dump(manifest, f, indent=1)
print(f"Panels indexed in {PUB_MANIFEST}")
try:
    from IPython.display import Image as _Img
    display(_Img(filename=static, width=900))
except Exception:
    pass

### Supplementary — channel coverage per animal

Which areas each animal contributes, and with how many sampled channels (the heatmaps
leave this out to stay minimal). Areas with one or two channels in most animals carry
noisier estimates.

In [ ]:
#@title Coverage table
for ctx, AGG in AGGS_ALL.items():
    banner(ctx)
    coverage = pd.DataFrame(AGG["n_ch_all"], columns=AGG["areas"], index=AGG["subjects"])
    coverage = coverage.loc[:, coverage.sum() > 0]
    display(coverage) if "display" in globals() else print(coverage.to_string())

### Supplementary — feature oddballs across contexts, side by side

One row per context that has the four feature oddballs (standard, sensorimotor, sequence)
and saved MI; the duration context has different oddball classes and keeps its own
figure. Same colour scale in every row, so contexts can be compared directly; grey cells
are areas not recorded in that context.

In [ ]:
#@title Comparison figure (needs at least two contexts)
FEATURES = list(BASE_LABELS)
AGGS = {c: g for c, g in AGGS_ALL.items()
        if set(FEATURES) <= {base_of(o) for o in g["oddballs"]}}
if len(AGGS) < 2:
    print(f"Only {list(AGGS) or 'no'} feature-oddball context(s) with MI so far.")
else:
    areas = [a for a in AREA_ORDER if any(a in g["show_areas"] for g in AGGS.values())]
    vmax = max(np.nanpercentile(g["mi_mean"][:, [g["areas"].index(a) for a in areas], :], 99.5)
               for g in AGGS.values())

    fig, axes = plt.subplots(len(AGGS), 4, figsize=(3.0 * 4 + 1.2, 4.6 * len(AGGS)),
                             sharey=True, constrained_layout=True, squeeze=False)
    for row, g in zip(axes, AGGS.values(), strict=True):
        odds = [next(o for o in g["oddballs"] if base_of(o) == b) for b in FEATURES]
        _, _, mesh = mi_heatmaps(g, axes=row, show_areas=areas, vmax=vmax, oddballs=odds,
                                 row_label=f"{g['label']} (n = {len(g['subjects'])})")
    cb = fig.colorbar(mesh, ax=axes, shrink=0.6, pad=0.01, extend="max")
    cb.set_label("MI (bits)")
    cb.outline.set_visible(False)
    fig.suptitle(f"Spectral information about prediction errors, by context "
                 f"(dots: FDR q < {ALPHA})", fontsize=11)
    out = context_dirs("standard")["figures"]
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out, f"oddball_mi_by_context_{MI_TAG}.{ext}"),
                    dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved to {out}")

## 7 · Interactive figure

Writes `publication/lfp-spectral-mi.html`: the same figure with a paradigm switch. By default
it loads the panel PNGs from `media/lfp-spectral-mi/` next to it (copied there), so each image
is stored only once; keep the two together when you move them. `HTML_EMBED = True` writes a
single self-contained file instead. It needs only the Storage cell: run that one, then this one.

In [ ]:
#@title Write the interactive figure (HTML + its panel images)
# Needs only the Storage cell: it reads the panels and index that section 6 saved.
# HTML_EMBED = False: the HTML loads the panels from HTML_MEDIA next to it (copied there), so
# each image is stored once (what the publication repository uses: interactive/ + media/).
# HTML_EMBED = True: one self-contained file with the images inside (~2x larger).
import base64
import json
import os
import shutil

HTML_EMBED = False                        #@param {type:"boolean"}
HTML_MEDIA = "media/lfp-spectral-mi"      # relative to the HTML file

PUB_DIR = os.path.join(RESULTS_DIR, "publication")
PUB_MANIFEST = os.path.join(PUB_DIR, "publication_manifest.json")
if not os.path.exists(PUB_MANIFEST):
    raise FileNotFoundError(f"{PUB_MANIFEST} not found — run section 6 first.")
with open(PUB_MANIFEST) as f:
    _man = json.load(f)
_media_dir = os.path.join(PUB_DIR, *HTML_MEDIA.split("/"))
os.makedirs(_media_dir, exist_ok=True)
for _m in _man.values():
    _m["images"] = {}
    for _k, _rel in _m["panels"].items():
        _src = os.path.join(PUB_DIR, _rel)
        if HTML_EMBED:
            with open(_src, "rb") as fh:
                _m["images"][_k] = "data:image/png;base64," + base64.b64encode(fh.read()).decode()
        else:
            shutil.copyfile(_src, os.path.join(_media_dir, os.path.basename(_src)))
            _m["images"][_k] = f"{HTML_MEDIA}/{os.path.basename(_src)}"

# Static/Interactive switch, "?" legend and its script follow the publication repository's
# shared viewer convention (figure_sources/javascript/figure-legend.css/.js, copied here).
FIGURE_LEGEND_CSS = '''.figure-toolbar {
  min-height: 56px;
  display: grid;
  grid-template-columns: minmax(0, 1fr) 36px;
  align-items: center;
  gap: 16px;
  padding: 10px 14px;
  border-bottom: 1px solid #d3d8d6;
  background: #fff;
}

.figure-mode-switch {
  justify-self: start;
  display: inline-grid;
  grid-template-columns: repeat(2, 88px);
  padding: 3px;
  gap: 2px;
  border: 1px solid #d3d8d6;
  border-radius: 6px;
  background: #f4f7f6;
}

.figure-mode-switch > button {
  position: static;
  min-width: 0;
  width: 88px;
  min-height: 30px;
  height: 30px;
  margin: 0;
  padding: 0 10px;
  border: 0;
  border-radius: 3px;
  background: transparent;
  color: #68716f;
  font: 650 14px "Myriad Pro", "Aptos", sans-serif;
  line-height: 1;
  letter-spacing: 0;
  white-space: nowrap;
  cursor: pointer;
}

.figure-mode-switch > button::after { content: none; }
.figure-mode-switch > button:hover { color: #293133; background: #e8eeeb; }
.figure-mode-switch > button[aria-pressed="true"] {
  color: #293133;
  background: #fff;
  box-shadow: 0 1px 3px #29313326;
}
.figure-mode-switch > button:focus-visible {
  outline: 2px solid #167f8c;
  outline-offset: 1px;
}

button.legend-toggle {
  width: 36px;
  height: 36px;
  min-width: 36px;
  flex: 0 0 36px;
  align-self: center;
  display: grid;
  place-items: center;
  margin: 0;
  padding: 0;
  border: 1px solid #8f9b98;
  border-radius: 50%;
  background: #fff;
  color: #293133;
  font-size: 22px;
  font-weight: 700;
  line-height: 1;
  cursor: pointer;
}

.figure-toolbar > .legend-toggle { grid-column: 2; justify-self: end; }

button.legend-toggle:hover,
button.legend-toggle[aria-expanded="true"] {
  border-color: #167f8c;
  background: #dceff0;
}

button.legend-toggle:focus-visible {
  outline: 3px solid #167f8c;
  outline-offset: 2px;
}

.figure-legend {
  min-width: 0;
  padding: 14px 22px;
  border-bottom: 1px solid #d3d8d6;
  background: #f4f7f6;
  color: #293133;
  font-size: var(--figure-type-control, 14px);
  line-height: 1.5;
}

.figure-legend h2 {
  margin: 0 0 8px;
  font-size: var(--figure-type-section-title, 16px);
  line-height: 1.3;
}

.figure-legend p { max-width: 85ch; margin: 0; }
.figure-legend p + p { margin-top: 8px; }
.figure-legend[hidden], .legend-toggle[hidden] { display: none !important; }

[data-simple-view] > [hidden] { display: none !important; }
[data-simple-view] #static-view img { width: 100%; height: auto; display: block; }

@media (max-width: 520px) {
  .figure-legend { padding: 12px 14px; }
}'''
FIGURE_LEGEND_JS = '''(() => {
  "use strict";

  const toggle = document.getElementById("figure-legend-toggle");
  const legend = document.getElementById("figure-legend");
  const staticButton = document.querySelector('[data-view="static"]');

  function setLegendOpen(open) {
    if (!toggle || !legend) return;
    legend.hidden = !open;
    toggle.setAttribute("aria-expanded", String(open));
  }

  function syncLegend() {
    if (!toggle || !staticButton) return;
    const isStatic = staticButton.getAttribute("aria-pressed") === "true"
      || staticButton.getAttribute("aria-selected") === "true";
    toggle.hidden = isStatic;
    if (isStatic) setLegendOpen(false);
  }

  if (toggle && legend) {
    toggle.addEventListener("click", () => setLegendOpen(legend.hidden));
    document.addEventListener("keydown", (event) => {
      if (event.key === "Escape" && !legend.hidden) {
        setLegendOpen(false);
        toggle.focus();
      }
    });
    if (staticButton) {
      new MutationObserver(syncLegend).observe(staticButton, {
        attributes: true,
        attributeFilter: ["aria-pressed", "aria-selected"],
      });
    }
    syncLegend();
  }

  const simpleViewer = document.querySelector("[data-simple-view]");
  if (simpleViewer) {
    const buttons = simpleViewer.querySelectorAll("[data-view]");
    function selectView(view) {
      simpleViewer.querySelector("#interactive-view").hidden = view !== "interactive";
      simpleViewer.querySelector("#static-view").hidden = view !== "static";
      buttons.forEach((button) => {
        const active = button.dataset.view === view;
        button.classList.toggle("active", active);
        button.setAttribute("aria-pressed", String(active));
      });
      syncLegend();
      window.dispatchEvent(new Event("resize"));
    }
    buttons.forEach((button) => {
      button.addEventListener("click", () => selectView(button.dataset.view));
    });
    selectView("static");
  }
})();'''
LEGEND_TEXT = (
    "<p><b>A,</b> one V1 trial (LFP below 100 Hz, aligned to stimulus onset; shading marks "
    "standards in blue and the event in its colour) above the spectrogram averaged over trials of "
    "the same event on the same channel. <b>B,</b> V1 power spectrum per condition, mean ± SEM "
    "across mice. <b>C,</b> the event and the standard (solid) next to the same stimuli in the "
    "matched control block (dashed); without a control, single-trial V1 power distributions. "
    "<b>D, E,</b> power difference in dB (red more, blue less) per area and frequency, event minus "
    "standard (D) or minus the same stimulus in the control block (E); dots mark cells where the "
    "mutual information between single-trial power and trial identity is significant across mice "
    "(FDR q &lt; 0.05).</p>"
    "<p>Mutual information asks how well the power of a single trial tells whether that trial "
    "was the event: 0 bits not at all, 1 bit perfectly; 0.04, 0.16 and 0.5 bits correspond to d′ "
    "of 0.5, 1 and 2. Event versus standard mixes a different stimulus with release from "
    "adaptation; the control block keeps the stimulus, and its orientation tuning, the same, and "
    "<i>Standard (adaptation)</i> compares the standard with its control.</p>")


def _panels_markup(m, prefix):
    # The A-E grid of one paradigm as static HTML (ids get a prefix).
    fig = lambda k, alt, cls="": (  # noqa: E731
        f'<figure class="lfp-panel {cls}" id="{prefix}fig-{k}"'
        + ("" if k in m["images"] else " hidden") + f'><span class="letter">{k}</span>'
        f'<img id="{prefix}img-{k}" alt="{alt}" src="{m["images"].get(k, "")}"></figure>')
    return ('<div class="lfp-top">'
            + fig("A", "Example V1 trial and trial-averaged spectrogram")
            + fig("B", "V1 power spectra per condition")
            + fig("C", "V1 spectra versus the matched control, or single-trial power")
            + "</div>" + fig("D", "Power difference versus the standard by area and frequency",
                             "wide")
            + fig("E", "Power difference versus the matched control by area and frequency",
                  "wide"))


_static_ctx = "standard" if "standard" in _man else next(iter(_man))
INTERACTIVE_HTML = '''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<meta name="color-scheme" content="light">
<title>LFP spectral information about mismatch events</title>
<style>
  :root { --figure-type-panel-title: 1.25rem; --figure-type-section-title: 1rem;
          --figure-type-body: 0.875rem; --figure-type-control: 0.8125rem;
          --figure-type-metadata: 0.75rem; --ink: #293133; --muted: #5f6b6a; --line: #d3d8d6;
          --accent: #167f8c; }
  body { margin: 0; background: #fff; color: var(--ink);
         font: var(--figure-type-body)/1.45 system-ui, -apple-system, "Segoe UI", sans-serif; }
  main.viewer { max-width: 1180px; margin: 0 auto; }
  __FIGURE_LEGEND_CSS__
  .lfp-body { padding: 12px 14px 18px; }
  .paradigms { display: inline-flex; flex-wrap: wrap; border: 1px solid var(--line);
               border-radius: 6px; overflow: hidden; margin: 0 0 10px; }
  .paradigms button { font: inherit; font-size: var(--figure-type-control); color: var(--ink);
                      background: #fff; border: 0; padding: 6px 12px; cursor: pointer; }
  .paradigms button + button { border-left: 1px solid var(--line); }
  .paradigms button[aria-pressed="true"] { background: var(--accent); color: #fff; }
  .paradigms button:focus-visible { outline: 2px solid var(--accent); outline-offset: -2px; }
  .lfp-note { color: var(--muted); margin: 0 0 10px; font-size: var(--figure-type-metadata); }
  .lfp-top { display: grid; grid-template-columns: 50fr 25fr 25fr; gap: 8px; align-items: start; }
  figure.lfp-panel { margin: 0; border: 1px solid var(--line); border-radius: 6px; padding: 4px;
                     position: relative; background: #fff; }
  figure.lfp-panel img { display: block; width: 100%; height: auto; }
  figure.lfp-panel .letter { position: absolute; left: 6px; top: 2px; font-weight: 700;
                             font-size: var(--figure-type-section-title); }
  figure.lfp-panel.wide { margin-top: 8px; }
  figure.lfp-panel[hidden] { display: none; }
  @media (max-width: 900px) { .lfp-top { grid-template-columns: 1fr 1fr; }
                              .lfp-top figure:first-child { grid-column: 1 / -1; } }
  @media (max-width: 560px) { .lfp-top { grid-template-columns: 1fr; } }
</style>
</head>
<body>
  <main class="viewer" id="lfp-spectral-mi-viewer" data-simple-view>
    <header class="figure-toolbar">
      <div class="figure-mode-switch" role="group" aria-label="Figure view">
        <button type="button" data-view="static" aria-pressed="true">Static</button>
        <button type="button" data-view="interactive" aria-pressed="false">Interactive</button>
      </div>
      <button type="button" class="legend-toggle" id="figure-legend-toggle" aria-label="Figure legend"
        title="Figure legend" aria-expanded="false" aria-controls="figure-legend" hidden>?</button>
    </header>
    <section class="figure-legend" id="figure-legend" aria-labelledby="figure-legend-title" hidden>
      <h2 id="figure-legend-title">Interactive figure legend</h2>
      __LEGEND_TEXT__
    </section>
    <div id="interactive-view" class="lfp-body" hidden>
      <div class="paradigms" id="ctx" role="group" aria-label="Paradigm"></div>
      <p class="lfp-note" id="note"></p>
      __INTERACTIVE_PANELS__
    </div>
    <section id="static-view" class="lfp-body" aria-label="Static LFP spectral information figure">
      __STATIC_PANELS__
    </section>
  </main>
<script>
__FIGURE_LEGEND_JS__
</script>
<script>
(() => {
  "use strict";
  const MAN = __MANIFEST__;
  const order = ["standard", "sequence", "sensorimotor", "duration"].filter((c) => MAN[c]);
  const bar = document.getElementById("ctx");
  function show(ctx) {
    const m = MAN[ctx];
    for (const b of bar.children) b.setAttribute("aria-pressed", String(b.dataset.ctx === ctx));
    for (const k of ["A", "B", "C", "D", "E"]) {
      const src = m.images[k];
      document.getElementById("i-fig-" + k).hidden = !src;
      if (src) document.getElementById("i-img-" + k).src = src;
    }
    document.getElementById("note").textContent =
      `${m.label} · ${m.n_animals} mice · V1 example: ${m.event_label}. ${m.note}`;
  }
  for (const c of order) {
    const b = document.createElement("button");
    b.type = "button"; b.dataset.ctx = c; b.textContent = MAN[c].label;
    b.addEventListener("click", () => show(c));
    bar.appendChild(b);
  }
  show(order[0]);
})();
</script>
</body>
</html>
'''
INTERACTIVE_HTML = (INTERACTIVE_HTML
                    .replace("__FIGURE_LEGEND_CSS__", FIGURE_LEGEND_CSS)
                    .replace("__FIGURE_LEGEND_JS__", FIGURE_LEGEND_JS)
                    .replace("__LEGEND_TEXT__", LEGEND_TEXT)
                    .replace("__STATIC_PANELS__", _panels_markup(_man[_static_ctx], "s-"))
                    .replace("__INTERACTIVE_PANELS__", _panels_markup(_man[_static_ctx], "i-")))

INTERACTIVE_FILE = os.path.join(PUB_DIR, "lfp-spectral-mi.html")
with open(INTERACTIVE_FILE, "w", encoding="utf-8", newline="\n") as f:
    f.write(INTERACTIVE_HTML.replace("__MANIFEST__", json.dumps(_man)))
print(f"Wrote {INTERACTIVE_FILE} ({os.path.getsize(INTERACTIVE_FILE) / 1e6:.2f} MB)")
if not HTML_EMBED:
    _pngs = sorted(f for f in os.listdir(_media_dir) if f.endswith(".png"))
    _mb = sum(os.path.getsize(os.path.join(_media_dir, f)) for f in _pngs) / 1e6
    print(f"Panels it loads: {len(_pngs)} PNGs, {_mb:.1f} MB, in {_media_dir}")
print("For the pull request: lfp-spectral-mi.html, " + (f"{HTML_MEDIA}/*.png, " if not HTML_EMBED
      else "") + "supplementary-lfp-spectral-mi.png")